# LEGO restoration - staged training (v5.0: one process per GPU)

============================================================================
LEGO restoration - staged, resumable training script (v5.0: DistributedDataParallel - one process per GPU - plus a per-GPU data layout)
Run it once per Kaggle session. Set CFG.STAGE below:
  STAGE 1  DIV2K (local, no internet).  Pixel-loss "RealESRNet-style" training.
  STAGE 2  Streaming COCO + Flickr30k (internet ON) MIXED with DIV2K replay,
           low LR + L2-SP anchor to the stage-1 weights  -> diversity WITHOUT
           catastrophic forgetting.
  STAGE 3  GAN fine-tune (internet ON for VGG19 weights + streaming):
           pixel + VGG perceptual + U-Net discriminator, like Real-ESRGAN.
           This is the stage that makes textures look sharp and natural.
v5.0 (the v4.5 log proved DataParallel was the bottleneck: 2 GPUs 5.5 s/step vs ONE GPU 1.55 s/step, GPUs 8-24 % busy):
  * DDP: one training process per GPU, gradients all-reduced (NCCL, gloo as fallback, single GPU as last fallback).
    Same global batch (2 x 4 = 8), same model / loss / LR schedule / checkpoints -> a v4.x checkpoint resumes as is.
    Expected: ~1.5-1.7 s/step instead of 6.1 s (about 4x), i.e. the remaining stage-1 steps take ~17 h instead of ~66 h.
  * DATA LAYOUT per GPU (CFG.DATA_LAYOUT): 'split' = GPU 0 trains on DIV2K while GPU 1 trains on streamed COCO+Flickr, in the
    SAME model, every step (gradients are averaged). This is stage-2's 50 % replay without a second stage.
    If the stream cannot be opened (Internet OFF) stage 1 falls back to DIV2K on both GPUs by itself.
  * All control decisions (time budget, RAM restart, NaN skip) are agreed between the GPUs, only rank 0 validates / saves / logs.
  * Different random seeds per GPU (a forked process would otherwise draw IDENTICAL crops on both GPUs).
  * Single-GPU fallback = batch 4 per step (batch 8 does not fit on one T4 with this model).
v4.2 SPEED (same model, same weights, same loss, same steps/epochs - only the maths is executed faster):
  * deformable convolution re-implemented with F.grid_sample + one fp16 tensor-core matmul (torchvision's
    deform_conv2d runs in fp32 under AMP and has a very slow backward). Verified against torchvision at start-up
    (outputs AND gradients), then benchmarked on YOUR GPU; the fastest equivalent variant is chosen automatically.
    If the self-test fails the notebook silently falls back to the old torchvision path (= v4.1 behaviour).
  * offset+mask convs fused into one tensor-core friendly conv (same weights, zero-padded to 32 channels)
  * EMA update via torch._foreach, no per-step .item() sync, bigger DataLoader prefetch, log shows recent it/s + ETA
Checkpoints / old weights stay 100% compatible (no parameter was renamed or reshaped).
v4.3 FIXES (model, losses and number of steps are unchanged):
  * LR schedule: the cosine cycles now fit each stage exactly and the LR reaches its minimum on the very LAST step
    (the log prints the cycle plan). Stage 3 is a single smooth decay (restarts are a bad idea for a GAN).
  * Streaming (stage 2/3): COCO and Flickr30k are read in parallel and mixed by CFG/STREAM_WEIGHTS; each session
    shuffles ALL shards with a fresh seed (the old .skip() pinned the order); validation images are excluded by hash.
  * Stage 1: every batch now draws its 8 crops from up to 8 different images (per-worker image pool) at the same
    decode cost; steps per epoch are unchanged.
  * fp16 deformable-conv path uses ~half the VRAM and is self-tested under autocast too.
v4.5 (diagnosis of the slow step; model, losses, steps unchanged):
  * STEP PHASES: for 6 synchronised steps at the start the loop times data-wait / host->GPU / forward+loss / backward /
    optimizer / EMA separately and prints where the seconds go.
  * GPU LINE every 100 steps: per-GPU utilisation, SM clock vs max clock, power, temperature, throttle reason,
    allocator retries (the signs of a throttled or memory-thrashing GPU).
  * PROFILE also times ONE GPU vs both GPUs through DataParallel and prints the DataParallel efficiency.
v4.4 (robustness + diagnosis; model, losses, steps unchanged):
  * RAM governor: trims the heap, deletes disposable files (old sample grids, stale .tmp, old stage-3 epoch weights) and,
    if RAM still climbs, saves and RESTARTS the training process automatically (exact resume, ~1 min) - the run no
    longer dies of memory. Restart also happens at an epoch end when RAM is above CFG.EPOCH_RESTART_FRAC.
  * Disk governor: output folder is kept under CFG.WORK_QUOTA_GB; before every save the oldest disposable files are
    deleted (resume files + best/latest weights are never touched); a full disk is caught and retried, not fatal.
  * Startup PROFILE (first start only): measures CPU decode/degrade cost, real DataLoader batches/s and GPU steps/s and
    prints a DATA-BOUND / GPU-BOUND verdict. The step log now shows how long each step WAITED for data.
  * RAM log line shows main / workers / shm / GPU-cache so a leak can be located.
  * Checkpoints remember the step inside the epoch (resume continues mid-epoch); DCN autotune result is cached.
Every stage pauses cleanly at MAX_SESSION_HOURS and writes ckpt_latest.pth.
NEXT SESSION: attach the previous notebook's Output as an input dataset and run
again - it finds the newest checkpoint automatically and resumes (same stage)
or hands over (next stage; EMA weights become the new init).
Memory hardening (your "tried to allocate more memory than is available"):
  * DataLoader workers are re-created every epoch (no slow fork/COW growth)
  * file list stored as a numpy array (avoids the refcount copy-on-write leak)
  * streaming uses compressed bytes + bounded queues + small shuffle buffer
  * glibc arena cap + malloc_trim, container-RAM watchdog that checkpoints and
    stops cleanly BEFORE the OOM-killer fires, RAM logged every ~100 steps
============================================================================

---
### How to use on Kaggle
1. **File > Import Notebook**, upload this `.ipynb`.
2. Session options: **Accelerator = GPU T4 x2**. Stage 2 and 3 (and the default 'split' layout of stage 1) want **Internet ON**; with Internet OFF stage 1 simply trains on DIV2K only.
3. **Add Input**: your DIV2K dataset (+ your old `lego-v2` weights dataset). For a later session also add the previous notebook's *Output*.
4. In the **Config** cell set `STAGE = 1` (then 2, then 3 in later sessions).
5. **Run All**. The `PREFLIGHT` block at the top of the log tells you exactly what the notebook sees.

In [1]:
# Needed for stage 2/3 only (streaming). Skipped silently if already installed / interfrom kaggle_secrets import UserSecretsClient 
import importlib.util, subprocess, sys
if importlib.util.find_spec("datasets") is None:
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], check=False, timeout=300)
    except Exception as e:
        print("datasets not installed (fine for stage 1):", e)
print("datasets available:", importlib.util.find_spec("datasets") is not None)

datasets available: True


In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")


## Imports & config  
**Set `STAGE` here (1, 2 or 3).**

In [3]:
import os, gc, sys, cv2, time, math, copy, queue, random, shutil, warnings, threading, traceback, itertools, zlib, json, glob, multiprocessing
import datetime, subprocess, socket
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
import torchvision
import torchvision.ops as ops
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
cv2.setNumThreads(0)
try:
    import ctypes
    _libc = ctypes.CDLL("libc.so.6")
    _libc.mallopt(-8, 2)            # M_ARENA_MAX = 2 : stops multi-thread heap bloat
    _libc.mallopt(-3, 1 << 20)      # M_MMAP_THRESHOLD = 1 MB (fixed): big temporary arrays go straight back to the OS
    _libc.mallopt(-1, 1 << 20)      # M_TRIM_THRESHOLD = 1 MB
except Exception:
    _libc = None
try:
    import psutil
except Exception:
    psutil = None


class CFG:
    STAGE = 2                       # <<< 1, 2 or 3 (see header)
    WORK_DIR = "/kaggle/working/"
    INPUT_DIR = "/kaggle/input/"
    MAX_SESSION_HOURS = 11.0

    EMBED_DIM = 128
    PATCH = 256
    BATCH_PATCHES = 16               # GLOBAL patches per optimiser step, split over the GPUs (2 GPUs -> 4 each). Stage 3 OOM? use 6
    MAX_BATCH_PER_GPU = 4           # one T4 (14.6 GB) holds 4 patches of 256 px with this model, not 8
    CROPS_PER_IMG = 4               # stage 1 / replay: crops per decoded DIV2K image
    CROPS_PER_STREAM_IMG = 1        # 1 crop per streamed image -> every stream patch of a batch is a different photo
    NUM_WORKERS = 4                 # DIV2K DataLoader workers PER GPU PROCESS (DIV2K PNG decode = 0.7 s/image)
    POOL_IMAGES = 16                # stage 1: decoded images kept per worker (crops of a batch come from different ones)
    POOL_WARM = 6                   # stage 1: decode every sample until this many images are in the pool
    STREAM_BUILDERS = 2             # streaming GPU process: threads that degrade + build batches
    PREFETCH = 3                    # batches prefetched per worker (small: ~1.5 uint8 MB per batch)
    DCN_BACKEND = "auto"            # "auto" (self-test + benchmark) | "gridsample" | "torchvision" (= old behaviour)
    DCN_CHECKPOINT = "auto"         # gridsample only: recompute the sampling in backward to save VRAM (True/False/"auto")
    FUSE_OFFSET_MASK = True         # one conv for offsets+mask instead of two (math identical)
    GRAD_CLIP = 1.0
    EMA_DECAY = 0.999
    WEIGHT_DECAY = 1e-3

    VAL_FRACTION = 0.05
    VAL_MAX_IMAGES = 40
    VAL_SEVERITY = 0.6              # fixed -> comparable validation curve
    STREAM_VAL_IMAGES = 24
    CKPT_EVERY_MIN = 15
    LOG_EVERY_STEPS = 25
    MIN_LAPVAR = 15.0               # reject blurry GROUND TRUTH crops from streamed data
    STREAM_SHUFFLE_BUFFER = 100
    RAM_WARN_FRAC = 0.60            # trim + delete disposable files
    RAM_STOP_FRAC = 0.75            # save a checkpoint and restart the training process (before the OOM killer)
    EPOCH_RESTART_FRAC = 0.50       # at an epoch end: restart the process if RAM is above this (clean + exact)
    SUPERVISED = True               # run training in a child process that is restarted automatically (see last cell)
    PIN_MEMORY = False              # DataLoader pin_memory (a batch is only ~3 MB, so this is not a bottleneck)
    PHASE_PROFILE = True            # time the phases of 6 real steps (first start only)
    PHASE_START = 4
    PHASE_STEPS = 6
    PROFILE_AT_START = True         # ~1-2 min diagnostic on the first start (see log: 'PROFILE' / 'VERDICT')
    WORK_QUOTA_GB = 17.0            # keep the output folder below this (Kaggle allows 20 GB)
    DISK_MIN_FREE_GB = 1.5
    KEEP_SAMPLES = 24               # newest validation sample grids to keep
    WARM_START_OLD = True           # stage 1 only: init from old LEGO_*.pth if no checkpoint
    VGG_WEIGHTS_PATH = ""           # optional: path to vgg19 state_dict if internet is off in stage 3
    FORGET_WARN_DB = 0.5

    # ---- v5.0 multi-GPU: one process per GPU
    USE_DDP = True                  # True: one process per GPU (DistributedDataParallel). False: single GPU (batch 4)
    MAX_GPUS = 2
    DDP_BACKENDS = ("nccl", "gloo") # tried in this order; if none starts -> automatic single-GPU fallback
    DDP_INIT_TIMEOUT_MIN = 5        # how long to wait for the GPUs to find each other at start
    DDP_TIMEOUT_MIN = 20            # later collectives (one GPU may validate / save / fetch while the other waits)
    NCCL_P2P_DISABLE = True         # T4 pairs on Kaggle have no NVLink: P2P can hang, going through host RAM costs ~30 ms/step
    CTRL_EVERY = 5                  # steps between the 'stop / restart?' agreement between the GPUs
    STREAM_VAL_TIMEOUT_S = 150      # cap for fetching the stream validation images (the other GPU waits for it)
    SEED = 1234
    # which data each GPU trains on:
    #   "auto"  : stage 1 -> split, stage 2 -> split, stage 3 -> mixed
    #   "div2k" : every GPU trains on DIV2K (no Internet needed)
    #   "split" : GPU 0 = DIV2K, GPU 1 = streamed COCO+Flickr (single GPU: one stream with 50 % DIV2K replay)
    #   "mixed" : every GPU = stream + STAGES[stage]['replay'] share of DIV2K  (stage 3 default)
    DATA_LAYOUT = "split"


STAGES = {
    1: dict(epochs=100, spe=None, lr=1.5e-4, warmup=300, t0=8, t_mult=2, rdecay=0.7,
            sev0=0.25, ramp=20, replay=0.0, l2sp=0.0, second_p=0.3, fft_w=0.1, grad_w=0.1),
    2: dict(epochs=60, spe=1500, lr=5e-5, warmup=200, t0=10, t_mult=2, rdecay=0.8,
            sev0=1.0, ramp=1, replay=0.5, l2sp=5e-4, second_p=0.6, fft_w=0.1, grad_w=0.1),
    3: dict(epochs=80, spe=1000, lr=5e-5, lr_d=5e-5, warmup=200, t0=80, t_mult=2, rdecay=0.8,   # t0=80: ONE cosine cycle
            sev0=1.0, ramp=1, replay=0.25, l2sp=2e-4, second_p=0.6, fft_w=0.02, grad_w=0.05,
            pix_w=1.0, perc_w=1.0, gan_w=0.1),
}

# (dataset, [split names to try in order])  - flickr30k on HF only has "test"
STREAM_SOURCES = [("detection-datasets/coco", ["train", "val", "validation"]),
                  ("lmms-lab/flickr30k", ["test", "train"])]
STREAM_WEIGHTS = [0.7, 0.3]         # share of streamed images per source (same order as STREAM_SOURCES)

LOG_FILE = os.path.join(CFG.WORK_DIR, "training_log.txt")
SAMPLES_DIR = os.path.join(CFG.WORK_DIR, "samples")
METRICS_CSV = os.path.join(CFG.WORK_DIR, "metrics.csv")
START_TIME = time.time()
IN_SUPERVISOR = False              # set True inside the supervised child process


class DIST:
    """Per-process distributed state (filled in by _rank_entry; the defaults = plain single-GPU run)."""
    on = False
    rank = 0
    world = 1
    local = 0
    backend = None
    attempt = 0
    bpr = None                      # patches per GPU per step (set in main)


def is_main():
    return DIST.rank == 0


def hours():
    return (time.time() - START_TIME) / 3600


def log(msg, all_ranks=False):
    """Only rank 0 prints (otherwise every line would appear once per GPU) unless all_ranks=True."""
    if DIST.rank != 0 and not all_ranks:
        return
    line = f"[{time.strftime('%H:%M:%S')}] ({hours():.2f}h) " + (f"[r{DIST.rank}] " if DIST.rank else "") + f"{msg}"
    print(line, flush=True)
    try:
        with open(LOG_FILE, "a") as f:
            f.write(line + "\n")
    except Exception:
        pass

## Multi-GPU helpers (agreement between the GPU processes)

In [4]:
def _flag_dev():
    return torch.device("cuda", DIST.local) if DIST.backend == "nccl" else torch.device("cpu")


def _reduce(value, op):
    t = torch.tensor([float(value)], device=_flag_dev())
    dist.all_reduce(t, op=op)
    return float(t.item())


def sync_max(value):
    """Lock-step decision: EVERY rank must call this at the same point of the loop. Returns the max over the ranks."""
    return _reduce(value, dist.ReduceOp.MAX) if DIST.on else float(value)


def sync_min(value):
    return _reduce(value, dist.ReduceOp.MIN) if DIST.on else float(value)


def gather_scalar(value):
    """One float per rank (collective: every rank must call it)."""
    if not DIST.on:
        return [float(value)]
    t = torch.zeros(DIST.world, device=_flag_dev())
    t[DIST.rank] = float(value)
    dist.all_reduce(t, op=dist.ReduceOp.SUM)
    return t.tolist()


def broadcast_obj(obj, src=0):
    """Send a small picklable object from rank `src` to all ranks (collective)."""
    if not DIST.on:
        return obj
    box = [obj if DIST.rank == src else None]
    dist.broadcast_object_list(box, src=src)
    return box[0]


def seed_everything(attempt=0):
    """A forked process inherits the parent's RNG state -> without this both GPUs would draw IDENTICAL crops."""
    s = CFG.SEED + 7919 * DIST.rank + 104729 * attempt
    random.seed(s)
    np.random.seed(s % (2 ** 32))
    torch.manual_seed(s)


## Memory watchdog

In [5]:
def trim_memory():
    gc.collect()
    if _libc is not None:
        try:
            _libc.malloc_trim(0)
        except Exception:
            pass


def _read_int(path):
    try:
        with open(path) as f:
            s = f.read().strip()
        return None if s == "max" else int(s)
    except Exception:
        return None


def ram_usage():
    """(used_GB, limit_GB) of the *container* (cgroup) if available, else of the host."""
    limit = used = None
    lim2 = _read_int("/sys/fs/cgroup/memory.max")
    if lim2:                                                      # cgroup v2
        limit = lim2
        try:
            with open("/sys/fs/cgroup/memory.stat") as f:
                st = dict(l.split() for l in f)
            used = int(st.get("anon", 0)) + int(st.get("shmem", 0))
        except Exception:
            used = _read_int("/sys/fs/cgroup/memory.current")
    else:
        lim1 = _read_int("/sys/fs/cgroup/memory/memory.limit_in_bytes")
        if lim1 and lim1 < 2 ** 50:                               # cgroup v1
            limit = lim1
            try:
                with open("/sys/fs/cgroup/memory/memory.stat") as f:
                    st = dict(l.split() for l in f)
                used = int(st.get("total_rss", 0)) + int(st.get("total_shmem", 0))
            except Exception:
                used = _read_int("/sys/fs/cgroup/memory/memory.usage_in_bytes")
    if (limit is None or used is None) and psutil is not None:
        vm = psutil.virtual_memory()
        limit, used = vm.total, vm.total - vm.available
    if limit is None or used is None:
        return 0.0, 1.0
    return used / 2 ** 30, limit / 2 ** 30


def ram_breakdown():
    """Where is the memory? main process / DataLoader workers / shared memory / GPU caching allocators."""
    parts = []
    try:
        if psutil is not None:
            me = psutil.Process()
            kids, kid_rss = me.children(recursive=True), 0
            for k in kids:
                try:
                    kid_rss += k.memory_info().rss
                except Exception:
                    pass
            parts.append(f"main {me.memory_info().rss / 2 ** 30:.1f} GB")
            parts.append(f"{len(kids)} children {kid_rss / 2 ** 30:.1f} GB")
    except Exception:
        pass
    try:
        st = os.statvfs("/dev/shm")
        parts.append(f"/dev/shm {(st.f_blocks - st.f_bfree) * st.f_frsize / 2 ** 30:.1f} GB")
    except Exception:
        pass
    try:
        if torch.cuda.is_available():
            parts.append(f"GPU{DIST.local} cache {torch.cuda.memory_reserved(DIST.local) / 2 ** 30:.1f} GB")
    except Exception:
        pass
    return " | ".join(parts)


def _dir_bytes(path):
    tot = 0
    for root, _, files in os.walk(path):
        for f in files:
            try:
                tot += os.path.getsize(os.path.join(root, f))
            except OSError:
                pass
    return tot


def _rm(path, why):
    try:
        sz = os.path.getsize(path)
        os.remove(path)
        log(f"cleanup: removed {os.path.relpath(path, CFG.WORK_DIR)} ({sz / 2 ** 20:.0f} MB) - {why}")
        return sz
    except OSError:
        return 0


def cleanup_workdir(level=1):
    """Frees space in the output folder, safest files first. NEVER touches what is needed to resume or the final
    weights: ckpt_latest.pth, LEGO_latest.pth, LEGO_s*_latest/_best.pth, metrics.csv, training_log.txt.
      level 0: stale *.tmp files            level 1: + old sample grids, old stage-3 epoch weights
      level 2: + ckpt_prev.pth (only if ckpt_latest.pth exists)"""
    W, freed = CFG.WORK_DIR, 0
    for p in glob.glob(os.path.join(W, "*.tmp")):
        freed += _rm(p, "stale partial file")
    if level >= 1:
        samples = sorted(glob.glob(os.path.join(W, "samples", "*.png")), key=os.path.getmtime)
        for p in samples[:max(0, len(samples) - CFG.KEEP_SAMPLES)]:
            freed += _rm(p, "old sample grid")
        eps = sorted(glob.glob(os.path.join(W, "LEGO_s3_ep*.pth")))
        for p in eps[:-2]:
            freed += _rm(p, "old stage-3 epoch weights")
    if level >= 2 and os.path.exists(os.path.join(W, "ckpt_latest.pth")):
        freed += _rm(os.path.join(W, "ckpt_prev.pth"), "previous checkpoint (space needed)")
    return freed


def make_room(need_bytes, what="file"):
    """Called before every save: keeps the output folder under WORK_QUOTA_GB and the disk above DISK_MIN_FREE_GB."""
    quota = CFG.WORK_QUOTA_GB * 2 ** 30

    def short():
        used = _dir_bytes(CFG.WORK_DIR)
        try:
            free = shutil.disk_usage(CFG.WORK_DIR).free
        except OSError:
            free = 1 << 60
        return (used + need_bytes > quota) or (free - need_bytes < CFG.DISK_MIN_FREE_GB * 2 ** 30), used, free

    for level in (0, 1, 2):
        bad, used, free = short()
        if not bad:
            return True
        cleanup_workdir(level)
    bad, used, free = short()
    if bad:
        log(f"!!! LOW SPACE: output folder {used / 2 ** 30:.1f} GB, disk free {free / 2 ** 30:.1f} GB - "
            f"trying to save the {what} anyway")
    return not bad


def ram_state():
    """returns 'ok' | 'warn' | 'stop' (= save + restart the process) plus a printable string."""
    used, limit = ram_usage()
    frac = used / max(limit, 1e-6)
    s = f"RAM {used:.1f}/{limit:.1f} GB ({frac * 100:.0f}%)"
    if frac >= CFG.RAM_WARN_FRAC:
        trim_memory()
        if DIST.rank == 0:
            cleanup_workdir(1)          # if the output folder lives in RAM this frees it
        used, limit = ram_usage()
        frac = used / max(limit, 1e-6)
        s = f"RAM {used:.1f}/{limit:.1f} GB ({frac * 100:.0f}%) after trim | " + ram_breakdown()
    if frac >= CFG.RAM_STOP_FRAC and used > 1.0:     # trust the container reading (the old host cross-check never fired)
        return "stop", s
    return ("warn" if frac >= CFG.RAM_WARN_FRAC else "ok"), s


## Degradations (Real-ESRGAN-style: blur/resize/noise/JPEG, 2nd order, sinc ringing)

In [6]:
try:
    from scipy.special import j1 as _j1
except Exception:
    _j1 = None


def _gauss(img, sev):
    s = random.uniform(0.4, 0.8 + 4.2 * sev)
    sy = s * random.uniform(0.7, 1.3)
    kx, ky = int(2 * math.ceil(3 * s) + 1), int(2 * math.ceil(3 * sy) + 1)
    return cv2.GaussianBlur(img, (kx, ky), sigmaX=s, sigmaY=sy)


def _motion(img, sev):
    hi = 7 + int(34 * sev)
    hi += (hi % 2 == 0)
    L = random.randrange(5, hi + 1, 2)
    ang = random.uniform(0, math.pi)
    c = L // 2
    k = np.zeros((L, L), np.float32)
    p1 = (int(round(c - math.cos(ang) * c)), int(round(c - math.sin(ang) * c)))
    p2 = (int(round(c + math.cos(ang) * c)), int(round(c + math.sin(ang) * c)))
    cv2.line(k, p1, p2, 1.0, 1)
    k /= k.sum()
    return cv2.filter2D(img, -1, k, borderType=cv2.BORDER_REFLECT)


def _defocus(img, sev):
    r = random.uniform(1.0, 1.5 + 5.0 * sev)
    h = int(math.ceil(r))
    yy, xx = np.mgrid[-h:h + 1, -h:h + 1]
    k = ((xx ** 2 + yy ** 2) <= r * r).astype(np.float32)
    k /= k.sum()
    return cv2.filter2D(img, -1, k, borderType=cv2.BORDER_REFLECT)


def _sinc(img, sev):
    """Circular low-pass sinc filter -> ringing / overshoot artefacts."""
    if _j1 is None:
        return img
    ks = random.choice([7, 9, 11, 13, 15, 17, 19, 21])
    wc = random.uniform(math.pi / 3, math.pi)
    ax = np.arange(ks) - ks // 2
    xx, yy = np.meshgrid(ax, ax)
    r = np.sqrt(xx ** 2 + yy ** 2)
    r[ks // 2, ks // 2] = 1.0
    k = wc * _j1(wc * r) / (2 * math.pi * r)
    k[ks // 2, ks // 2] = wc ** 2 / (4 * math.pi)
    if abs(k.sum()) < 1e-3:
        return img
    k = (k / k.sum()).astype(np.float32)
    out = cv2.filter2D(img.astype(np.float32), -1, k, borderType=cv2.BORDER_REFLECT)
    return np.clip(out, 0, 255).astype(np.uint8)


def _resample(img, sev):
    s = random.uniform(max(0.3, 0.9 - 0.55 * sev), 0.95)
    h, w = img.shape[:2]
    small = cv2.resize(img, (max(8, int(w * s)), max(8, int(h * s))),
                       interpolation=random.choice([cv2.INTER_AREA, cv2.INTER_LINEAR, cv2.INTER_CUBIC]))
    return cv2.resize(small, (w, h), interpolation=random.choice([cv2.INTER_LINEAR, cv2.INTER_CUBIC]))


def _noise(img, sev):
    h, w = img.shape[:2]
    if random.random() < 0.35:     # Poisson (shot) noise
        f = img.astype(np.float32) / 255.0
        vals = float(2 ** random.choice([4, 5, 6, 7]))
        noisy = np.random.poisson(f * vals).astype(np.float32) / vals
        out = f + (noisy - f) * random.uniform(0.5, 1.0 + 2.0 * sev)
        return np.clip(out * 255.0, 0, 255).astype(np.uint8)
    sigma = random.uniform(1.0, 3.0 + 22.0 * sev)
    n = np.random.randn(h, w, 1 if random.random() < 0.5 else 3).astype(np.float32) * sigma
    return np.clip(img.astype(np.float32) + n, 0, 255).astype(np.uint8)


def _jpeg(img, sev):
    q = random.randint(max(10, int(92 - 72 * sev)), 95)
    ok, buf = cv2.imencode(".jpg", cv2.cvtColor(img, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, q])
    if not ok:
        return img
    return cv2.cvtColor(cv2.imdecode(buf, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)


def _round(img, sev):
    r = random.random()
    if r < 0.9:
        img = _gauss(img, sev) if r < 0.36 else _motion(img, sev) if r < 0.68 else _defocus(img, sev)
    if random.random() < 0.30:
        img = _resample(img, sev)
    if random.random() < 0.20 + 0.40 * sev:
        img = _noise(img, sev)
    if random.random() < 0.30 + 0.40 * sev:
        img = _jpeg(img, sev)
    return img


def degrade(img, sev, second_p=0.3):
    """img: uint8 RGB HxWx3, sev in [0,1] -> degraded uint8 RGB, same size."""
    if random.random() < 0.05:           # a few untouched samples: "do not wreck sharp images"
        return img.copy()
    out = _round(np.ascontiguousarray(img), sev)
    if random.random() < second_p:       # second-order degradation
        out = _round(out, sev * 0.8)
    if random.random() < 0.12:
        out = _sinc(out, sev)
    if random.random() < 0.20:
        out = _jpeg(out, sev)
    return out
# %% END-DEGRADE

## Data

In [7]:
def pad_to(img, size):
    h, w = img.shape[:2]
    ph, pw = max(0, size - h), max(0, size - w)
    if ph or pw:
        img = cv2.copyMakeBorder(img, 0, ph, 0, pw, cv2.BORDER_REFLECT_101)
    return img


def random_crop_aug(img, size):
    """Crop + flip/rot applied to the CLEAN patch BEFORE degrading -> always pixel-aligned."""
    h, w = img.shape[:2]
    y, x = random.randint(0, h - size), random.randint(0, w - size)
    p = img[y:y + size, x:x + size]
    if random.random() < 0.5:
        p = p[:, ::-1]
    if random.random() < 0.2:
        p = p[::-1]
    if random.random() < 0.3:
        p = np.rot90(p, random.choice([1, 3]))
    return np.ascontiguousarray(p)


def lap_var_np(rgb):
    return float(cv2.Laplacian(cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY), cv2.CV_64F).var())


def to_u8_tensor(list_hwc):
    return torch.from_numpy(np.stack(list_hwc)).permute(0, 3, 1, 2).contiguous()   # uint8 (N,3,H,W)


def read_rgb(path, patch):
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:
        return None
    return pad_to(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), patch)


def scan_input(input_dir):
    """One pass over /kaggle/input: usable images, images skipped by the name rule, weight files."""
    info = dict(images=[], excluded=0, per_dir={}, weights=[])
    for root, _, files in os.walk(input_dir):
        low = root.lower()
        excl = ("lego" in low) or ("samples" in low)      # checkpoint datasets / saved sample grids
        n = 0
        for f in files:
            fl, full = f.lower(), os.path.join(root, f)
            if fl.endswith((".png", ".jpg", ".jpeg")):
                if excl:
                    info["excluded"] += 1
                else:
                    info["images"].append(full)
                    n += 1
            elif fl.endswith((".pth", ".pt")):
                try:
                    info["weights"].append((full, os.path.getsize(full) / 2 ** 20))
                except OSError:
                    pass
        if n:
            info["per_dir"][root] = n
    return info


def list_inputs(input_dir, depth=4):
    out = []
    base = input_dir.rstrip("/").count("/")
    for root, dirs, _ in os.walk(input_dir):
        if root.count("/") - base >= depth:
            dirs[:] = []
            continue
        out.append(root)
    return out


def preflight(info):
    """Prints WHAT THE SCRIPT SEES, so a run that cannot work fails loudly with the reason."""
    log("---------------- PREFLIGHT ----------------")
    log(f"python {sys.version.split()[0]} | torch {torch.__version__} | torchvision {torchvision.__version__} | "
        f"cv2 {cv2.__version__} | CPUs {os.cpu_count()}")
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            pr = torch.cuda.get_device_properties(i)
            log(f"GPU {i}: {pr.name} {pr.total_memory / 2 ** 30:.1f} GB")
    else:
        log("GPU: NONE")
    used, limit = ram_usage()
    host = f"{psutil.virtual_memory().available / 2 ** 30:.1f} GB free of host" if psutil else "psutil n/a"
    log(f"RAM (container reader): {used:.1f}/{limit:.1f} GB | {host}")
    log(f"/kaggle/input folders: {list_inputs(CFG.INPUT_DIR)[:20]}")
    log(f"usable images: {len(info['images'])} | skipped by name rule ('lego'/'samples' in path): {info['excluded']}")
    for d, n in sorted(info["per_dir"].items(), key=lambda kv: -kv[1])[:5]:
        log(f"   {n:5d} images in {d}")
    for w, mb in info["weights"][:8]:
        log(f"   weights file: {w} ({mb:.0f} MB)")
    log("-------------------------------------------")
    if not torch.cuda.is_available():
        raise RuntimeError("No GPU visible. In the notebook: Session options > Accelerator > GPU T4 x2, then re-run.")
    if not info["images"]:
        raise RuntimeError(
            f"NO TRAINING IMAGES FOUND under {CFG.INPUT_DIR} (skipped by name rule: {info['excluded']}). "
            "Attach your DIV2K dataset via 'Add Input'. Folders seen: " + str(list_inputs(CFG.INPUT_DIR)[:20]))


def to_model_range(u8):
    if u8.ndim == 5:
        u8 = u8.flatten(0, 1)
    return u8.float().div_(127.5).sub_(1.0)


class Stage1Dataset(Dataset):
    """Image-pool sampler. Each worker keeps POOL_IMAGES decoded images in RAM and swaps ONE of them every
    CROPS_PER_IMG samples, so the decode cost per batch is the same as before (BATCH/CROPS images) but the crops of
    a batch come from up to BATCH different images instead of 2. len() = images x crops -> steps/epoch unchanged."""
    def __init__(self, paths, patch, crops, severity, second_p, length=None):
        self.paths = np.array(paths)            # numpy, NOT a python list (avoids COW refcount leak)
        self.patch, self.crops, self.severity, self.second_p = patch, crops, severity, second_p
        self.length = int(length) if length else len(self.paths) * self.crops    # samples per epoch of THIS GPU
        self._pool, self._tick = None, 0        # created lazily INSIDE each worker process

    def __len__(self):
        return self.length

    def _decode_one(self):
        for _ in range(5):
            img = read_rgb(self.paths[random.randrange(len(self.paths))], self.patch)
            if img is not None:
                return img
        return np.full((self.patch, self.patch, 3), 127, np.uint8)

    def __getitem__(self, idx):
        if self._pool is None:
            self._pool, self._tick = [], 0
        self._tick += 1
        if len(self._pool) < CFG.POOL_WARM or self._tick % self.crops == 0:
            img = self._decode_one()
            if len(self._pool) < CFG.POOL_IMAGES:
                self._pool.append(img)
            else:
                self._pool[random.randrange(len(self._pool))] = img
        c = random_crop_aug(random.choice(self._pool), self.patch)
        d = degrade(c, random.uniform(0.0, float(self.severity[0])), self.second_p)
        return (torch.from_numpy(np.ascontiguousarray(d.transpose(2, 0, 1))),
                torch.from_numpy(np.ascontiguousarray(c.transpose(2, 0, 1))))


def worker_init(worker_id):
    np.random.seed((torch.initial_seed() + worker_id) % (2 ** 32))
    cv2.setNumThreads(0)


def center_crop(img, size):
    h, w = img.shape[:2]
    y, x = (h - size) // 2, (w - size) // 2
    return np.ascontiguousarray(img[y:y + size, x:x + size])


def make_val_set(clean_list, sev, seed0):
    """Fixed validation pairs. MUST be called before worker/feeder threads start (touches global RNG)."""
    py, npst = random.getstate(), np.random.get_state()
    degs = []
    for i, c in enumerate(clean_list):
        random.seed(seed0 + i)
        np.random.seed(seed0 + i)
        degs.append(degrade(c, sev, 0.5))
    random.setstate(py)
    np.random.set_state(npst)
    return to_u8_tensor(degs), to_u8_tensor(clean_list)


# ---- streaming (stage 2/3): feeder thread -> raw queue -> builder threads -> batch queue
def open_stream(src_i, skip=0, seed=0, buffer=0):
    from datasets import load_dataset, Image as HFImage
    name, splits = STREAM_SOURCES[src_i]
    last = None
    for sp in splits:
        try:
            ds = load_dataset(name, split=sp, streaming=True,token=HF_TOKEN)
            try:
                ds = ds.cast_column("image", HFImage(decode=False))   # keep compressed bytes (RAM!)
            except Exception:
                pass
            if skip:
                ds = ds.skip(skip)
            if buffer:
                ds = ds.shuffle(seed=seed, buffer_size=buffer)
            return ds
        except Exception as e:
            last = e
    raise RuntimeError(f"cannot open {name}: {last}")


def extract_raw(ex):
    im = ex.get("image")
    if isinstance(im, dict):
        b = im.get("bytes")
        return bytes(b) if b else None
    if im is not None and hasattr(im, "convert"):
        return np.array(im.convert("RGB"))
    return None


def decode_raw(item, patch, prescale):
    if isinstance(item, (bytes, bytearray)):
        a = cv2.imdecode(np.frombuffer(item, np.uint8), cv2.IMREAD_COLOR)
        if a is None:
            return None
        a = cv2.cvtColor(a, cv2.COLOR_BGR2RGB)
    else:
        a = item
    h, w = a.shape[:2]
    m = max(h, w)
    if m > 2048:
        a = cv2.resize(a, (int(w * 2048 / m), int(h * 2048 / m)), interpolation=cv2.INTER_AREA)
        h, w = a.shape[:2]
    if prescale:      # shrinking hides the source's own JPEG artefacts -> cleaner ground truth
        s = random.uniform(0.55, 0.9)
        if min(h, w) * s >= patch:
            a = cv2.resize(a, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA)
    return pad_to(np.ascontiguousarray(a), patch)


STREAM_VAL_HASH = set()      # (size, crc) of the stream images used for validation -> never trained on


def _raw_hash(item):
    if isinstance(item, (bytes, bytearray)):
        return (len(item), zlib.crc32(bytes(item[:65536])))
    a = np.ascontiguousarray(item)
    return (int(a.size), zlib.crc32(a.tobytes()[:65536]))


def fetch_stream_val(patch, n):
    """First images of EACH (unshuffled) stream -> deterministic stream validation crops. Their hashes are remembered
    so the shuffled training stream skips them (no train/val leak, and no .skip() that would pin the stream order)."""
    n_src = len(STREAM_SOURCES)
    per = max(1, -(-n // n_src))
    crops, errs = [], []
    t_end = time.time() + CFG.STREAM_VAL_TIMEOUT_S
    for src in range(n_src):
        sharp, soft, scanned = [], [], 0
        try:
            for ex in open_stream(src):
                if time.time() > t_end:
                    errs.append("timeout while fetching validation images")
                    break
                raw = extract_raw(ex)
                img = decode_raw(raw, patch, prescale=False) if raw is not None else None
                if img is None:
                    continue
                scanned += 1
                c = center_crop(img, patch)
                (sharp if lap_var_np(c) >= CFG.MIN_LAPVAR else soft).append((c, _raw_hash(raw)))
                if len(sharp) >= per or scanned >= 8 * per:
                    break
        except Exception as e:
            errs.append(str(e)[:150])
            log(f"stream-val source {src} failed: {errs[-1]}")
        chosen = sharp[:per] if len(sharp) >= per else (sharp + soft)[:per]
        for c, h in chosen:
            crops.append(c)
            STREAM_VAL_HASH.add(h)
        log(f"stream-val source {src} ({STREAM_SOURCES[src][0]}): {len(chosen)} images")
    if not crops:
        raise RuntimeError("Could not stream any images. Turn Internet ON in the notebook settings "
                           "(right sidebar > Session options). Errors: " + " | ".join(errs))
    return crops[:n]


class StreamFeeder(threading.Thread):
    """One thread per source (COCO, Flickr, ...). Each re-opens its stream with a NEW shuffle seed when it runs dry, so
    a source never starves the others and no session replays the same order."""
    def __init__(self, src, raw_q, stop, seed):
        super().__init__(daemon=True)
        self.src, self.raw_q, self.stop, self.seed = src, raw_q, stop, seed

    def run(self):
        name = STREAM_SOURCES[self.src][0]
        fails, rnd = 0, 0
        while not self.stop.is_set():
            try:
                ds = open_stream(self.src, seed=self.seed + 7919 * rnd, buffer=CFG.STREAM_SHUFFLE_BUFFER)
                if rnd == 0:
                    log(f"stream source opened: {name}", all_ranks=True)
                rnd += 1
                got = 0
                for ex in ds:
                    if self.stop.is_set():
                        return
                    item = extract_raw(ex)
                    if item is None or _raw_hash(item) in STREAM_VAL_HASH:
                        continue
                    while not self.stop.is_set():
                        try:
                            self.raw_q.put(item, timeout=1)
                            break
                        except queue.Full:
                            pass
                    got += 1
                    fails = 0
                if got == 0:
                    raise RuntimeError("stream returned no images")
            except Exception as e:
                fails += 1
                log(f"stream feeder [{name}] error (#{fails}): {str(e)[:200]} - retrying", all_ranks=True)
                self.stop.wait(min(60, 5 * fails))


class BatchBuilder(threading.Thread):
    def __init__(self, raw_qs, out_q, stop, train_paths, severity, scfg):
        super().__init__(daemon=True)
        self.raw_qs, self.out_q, self.stop = raw_qs, out_q, stop
        self.paths, self.severity, self.scfg = list(train_paths), severity, scfg

    def _crops(self, img, n, sharp_check):
        out = []
        for _ in range(n):
            for _try in range(5 if sharp_check else 1):
                c = random_crop_aug(img, CFG.PATCH)
                if not sharp_check or lap_var_np(c) >= CFG.MIN_LAPVAR:
                    break
            out.append(c)
        return out

    def _next_raw(self, timeout=15.0):
        """Pick a source by STREAM_WEIGHTS; if its queue is empty use the others; None after `timeout` s."""
        n, t_end = len(self.raw_qs), time.time() + timeout
        while not self.stop.is_set():
            first = random.choices(range(n), weights=STREAM_WEIGHTS[:n])[0]
            for j in [first] + [k for k in range(n) if k != first]:
                try:
                    return self.raw_qs[j].get_nowait()
                except queue.Empty:
                    pass
            if time.time() >= t_end:
                return None
            time.sleep(0.02)
        return None

    def run(self):
        total = DIST.bpr or CFG.BATCH_PATCHES          # patches per GPU per step
        n_rep = int(round(total * self.scfg["replay"]))
        n_str = total - n_rep
        stalled = False
        while not self.stop.is_set():
            try:
                cleans = []
                while len(cleans) < n_str and not self.stop.is_set():
                    item = self._next_raw()
                    if item is None:
                        if not stalled:
                            log("stream stalled - filling with DIV2K replay until it recovers", all_ranks=True)
                            stalled = True
                        break
                    stalled = False
                    img = decode_raw(item, CFG.PATCH, prescale=True)
                    if img is not None:
                        cleans += self._crops(img, min(CFG.CROPS_PER_STREAM_IMG, n_str - len(cleans)), True)
                need = total - len(cleans)
                while need > 0:                                   # replay of stage-1 data
                    img = read_rgb(random.choice(self.paths), CFG.PATCH)
                    if img is None:
                        continue
                    k = min(CFG.CROPS_PER_IMG, need)
                    cleans += self._crops(img, k, False)
                    need -= k
                smax = float(self.severity[0])
                degs = [degrade(c, random.uniform(0.0, smax), self.scfg["second_p"]) for c in cleans]
                batch = (to_u8_tensor(degs), to_u8_tensor(cleans))
                while not self.stop.is_set():
                    try:
                        self.out_q.put(batch, timeout=1)
                        break
                    except queue.Full:
                        pass
            except Exception as e:
                log(f"batch builder error: {str(e)[:200]}", all_ranks=True)
                self.stop.wait(2)


class MixedFeeder:
    def __init__(self, train_paths, severity, scfg, n_builders=None):
        n_builders = n_builders or CFG.STREAM_BUILDERS
        self.stop = threading.Event()
        n_src = len(STREAM_SOURCES)
        self.raw_qs = [queue.Queue(maxsize=24) for _ in range(n_src)]
        self.out_q = queue.Queue(maxsize=6)
        seed = (int(time.time()) + 10007 * DIST.rank + 31 * DIST.attempt) % 100000   # a different shuffle per GPU / restart
        self.threads = [StreamFeeder(i, self.raw_qs[i], self.stop, seed + 13 * i) for i in range(n_src)] + \
                       [BatchBuilder(self.raw_qs, self.out_q, self.stop, train_paths, severity, scfg)
                        for _ in range(n_builders)]
        for t in self.threads:
            t.start()

    def batches(self):
        while not self.stop.is_set():
            try:
                b = self.out_q.get(timeout=120)
            except queue.Empty:
                log("data stall (no batch for 120 s)", all_ranks=True)
                continue
            yield b

    def close(self):
        self.stop.set()
# %% END-DATA

## Model (IDENTICAL to your old architecture -> old weights load)

In [8]:
class DCN:
    """Runtime switches for the deformable-conv implementation (set once by autotune_dcn())."""
    backend = "torchvision"          # "torchvision" | "gridsample"
    checkpoint = False               # gridsample only


def deform_conv_gridsample(x, offsets, masks, weight, padding=1):
    """Exact re-implementation of torchvision.ops.deform_conv2d for the 3x3 / stride 1 / dilation 1 / 1 group /
    modulated (DCNv2) case: bilinear sampling (zeros outside the image) with F.grid_sample in fp32 (coordinates stay
    exact), then ONE matmul over (C*9) -> fp16 tensor cores under autocast. No parameter is touched."""
    B, C, H, W = x.shape
    O, _, kh, kw = weight.shape
    K = kh * kw
    dt = x.dtype
    xf, off, msk = x.float(), offsets.float(), masks.float()
    ys = torch.arange(H, device=x.device, dtype=torch.float32).view(1, H, 1) - padding
    xs = torch.arange(W, device=x.device, dtype=torch.float32).view(1, 1, W) - padding
    sy, sx = 2.0 / max(H - 1, 1), 2.0 / max(W - 1, 1)
    cols = []
    for k in range(K):
        i, j = divmod(k, kw)
        py = ys + i + off[:, 2 * k]
        px = xs + j + off[:, 2 * k + 1]
        grid = torch.stack((px * sx - 1.0, py * sy - 1.0), dim=-1)
        s = F.grid_sample(xf, grid, mode="bilinear", padding_mode="zeros", align_corners=True)
        cols.append(s.to(dt) * msk[:, k:k + 1].to(dt))          # cast first: backward keeps fp16 (not fp32) copies
    col = torch.stack(cols, dim=2).reshape(B, C * K, H * W)          # channel index = c*K + k (= weight layout)
    out = torch.matmul(weight.reshape(O, C * K), col)
    return out.reshape(B, O, H, W)


class DeformableConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, padding=1):
        super().__init__()
        self.kernel_size, self.padding = kernel_size, padding
        self.offset_conv = nn.Conv2d(in_channels, 2 * kernel_size * kernel_size, 3, padding=1)
        self.mask_conv = nn.Conv2d(in_channels, kernel_size * kernel_size, 3, padding=1)
        self.regular_conv = nn.Conv2d(in_channels, out_channels, kernel_size, padding=padding, bias=False)
        self.weight = self.regular_conv.weight
        for m in (self.offset_conv, self.mask_conv):
            nn.init.constant_(m.weight, 0.0)
            nn.init.constant_(m.bias, 0.0)

    def _offset_mask(self, x):
        """offset (18 ch) + mask (9 ch) from ONE conv: weights concatenated at run time and zero-padded to a multiple
        of 8 output channels (tensor-core friendly). Parameters keep their original names/shapes."""
        if not CFG.FUSE_OFFSET_MASK:
            return self.offset_conv(x), self.mask_conv(x)
        n_off, n_msk = self.offset_conv.out_channels, self.mask_conv.out_channels
        w = torch.cat([self.offset_conv.weight, self.mask_conv.weight], 0)
        b = torch.cat([self.offset_conv.bias, self.mask_conv.bias], 0)
        pad = (-(n_off + n_msk)) % 8
        if pad:
            w = F.pad(w, (0, 0, 0, 0, 0, 0, 0, pad))
            b = F.pad(b, (0, pad))
        y = F.conv2d(x, w, b, padding=1)
        return y[:, :n_off], y[:, n_off:n_off + n_msk]

    def forward(self, x):
        off_raw, msk_raw = self._offset_mask(x)
        offsets = torch.clamp(off_raw, -32.0, 32.0)
        masks = torch.sigmoid(msk_raw)
        if DCN.backend == "gridsample":
            if DCN.checkpoint and self.training and torch.is_grad_enabled():
                from torch.utils.checkpoint import checkpoint as _ckpt
                return _ckpt(deform_conv_gridsample, x, offsets, masks, self.weight, self.padding,
                             use_reentrant=False, preserve_rng_state=False)
            return deform_conv_gridsample(x, offsets, masks, self.weight, self.padding)
        return ops.deform_conv2d(x, offsets, self.weight, mask=masks, padding=self.padding)


class TransposedAttentionBlock(nn.Module):
    def __init__(self, channels, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.temperature = nn.Parameter(torch.ones(num_heads, 1, 1))
        self.qkv = nn.Conv2d(channels, channels * 3, 1, bias=False)
        self.qkv_dwconv = nn.Conv2d(channels * 3, channels * 3, 3, 1, 1, groups=channels * 3, bias=False)
        self.project_out = nn.Conv2d(channels, channels, 1, bias=False)

    def forward(self, x):
        b, c, h, w = x.shape
        q, k, v = self.qkv_dwconv(self.qkv(x)).chunk(3, dim=1)
        q = q.reshape(b, self.num_heads, c // self.num_heads, h * w)
        k = k.reshape(b, self.num_heads, c // self.num_heads, h * w)
        v = v.reshape(b, self.num_heads, c // self.num_heads, h * w)
        q, k = F.normalize(q, dim=-1), F.normalize(k, dim=-1)
        attn = ((q @ k.transpose(-2, -1)) * self.temperature).softmax(dim=-1)
        return x + self.project_out((attn @ v).reshape(b, c, h, w))


class DeepSpatialRestorationNet(nn.Module):
    def __init__(self, embed_dim=128):
        super().__init__()
        e = embed_dim
        self.initial = nn.Conv2d(3, e, 3, padding=1)
        self.enc1 = nn.Sequential(DeformableConvBlock(e, e), TransposedAttentionBlock(e))
        self.down1 = nn.Conv2d(e, e * 2, 4, stride=2, padding=1)
        self.enc2 = nn.Sequential(DeformableConvBlock(e * 2, e * 2), TransposedAttentionBlock(e * 2))
        self.down2 = nn.Conv2d(e * 2, e * 4, 4, stride=2, padding=1)
        self.bottleneck = nn.Sequential(*[TransposedAttentionBlock(e * 4) for _ in range(4)])
        self.up2 = nn.ConvTranspose2d(e * 4, e * 2, 2, stride=2)
        self.dec2 = nn.Sequential(DeformableConvBlock(e * 4, e * 2), TransposedAttentionBlock(e * 2))
        self.up1 = nn.ConvTranspose2d(e * 2, e, 2, stride=2)
        self.dec1 = nn.Sequential(DeformableConvBlock(e * 2, e), TransposedAttentionBlock(e))
        self.final = nn.Conv2d(e, 3, 3, padding=1)

    def forward(self, x):
        e1 = self.enc1(self.initial(x))
        e2 = self.enc2(self.down1(e1))
        b = self.bottleneck(self.down2(e2))
        d2 = self.dec2(torch.cat([self.up2(b), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return x + self.final(d1)

## Fast deformable conv: self-test + autotune (runs once at start of main)


In [9]:
def _dcn_selftest(device, amp=False):
    """torchvision vs gridsample on random data (offsets large enough to leave the image): forward output AND gradients
    w.r.t. input / weight / offsets / mask must agree. amp=True feeds fp16 activations under autocast (the real case)."""
    B, C, O, H, W = 2, 12, 10, 24, 20
    x = torch.randn(B, C, H, W, device=device, requires_grad=True)
    weight = torch.randn(O, C, 3, 3, device=device, requires_grad=True)
    offsets = (torch.randn(B, 18, H, W, device=device) * 3.0).clamp(-32, 32).requires_grad_(True)
    masks = torch.rand(B, 9, H, W, device=device).requires_grad_(True)
    probe = torch.linspace(0.5, 1.5, B * O * H * W, device=device).view(B, O, H, W)

    def ref():
        xi = x.half().float() if amp else x
        return ops.deform_conv2d(xi, offsets, weight, mask=masks, padding=1)

    def cand():
        if amp:
            with torch.autocast("cuda", dtype=torch.float16):
                return deform_conv_gridsample(x.half(), offsets, masks, weight, 1).float()
        return deform_conv_gridsample(x, offsets, masks, weight, 1)

    leaves, res = (x, weight, offsets, masks), []
    for fn in (ref, cand):
        for t in leaves:
            t.grad = None
        out = fn()
        (out * probe).sum().backward()
        res.append([out.detach().clone()] + [t.grad.detach().clone() for t in leaves])
    errs = [((a - b).abs().max() / (a.abs().max() + 1e-6)).item() for a, b in zip(*res)]
    return max(errs), errs


def _fuse_selftest(device):
    """fused offset+mask conv must equal the two separate convs."""
    blk = DeformableConvBlock(8, 8).to(device)
    for m in (blk.offset_conv, blk.mask_conv):
        nn.init.normal_(m.weight, std=0.1)
        nn.init.normal_(m.bias, std=0.1)
    x = torch.randn(2, 8, 16, 16, device=device)
    keep = CFG.FUSE_OFFSET_MASK
    try:
        CFG.FUSE_OFFSET_MASK = True
        o1, m1 = blk._offset_mask(x)
        CFG.FUSE_OFFSET_MASK = False
        o2, m2 = blk._offset_mask(x)
    finally:
        CFG.FUSE_OFFSET_MASK = keep
    return max((o1 - o2).abs().max().item(), (m1 - m2).abs().max().item())


def _bench_dcn(device, use_amp, batch, iters):
    """median seconds for one forward+backward of the real network at the per-GPU batch size, and peak VRAM (GB)."""
    net = x = out = None
    try:
        net = DeepSpatialRestorationNet(CFG.EMBED_DIM).to(device).train()
        for m in net.modules():              # non-trivial offsets -> realistic memory-access pattern
            if isinstance(m, DeformableConvBlock):
                nn.init.normal_(m.offset_conv.weight, std=0.01)
                nn.init.normal_(m.mask_conv.weight, std=0.01)
        x = torch.randn(batch, 3, CFG.PATCH, CFG.PATCH, device=device)
        torch.cuda.reset_peak_memory_stats(device)
        times = []
        for _ in range(iters + 1):           # first iteration = warm-up (cudnn autotune), not counted
            torch.cuda.synchronize(device)
            t0 = time.time()
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                out = net(x)
            out.float().abs().mean().backward()
            net.zero_grad(set_to_none=True)
            out = None
            torch.cuda.synchronize(device)
            times.append(time.time() - t0)
        return float(np.median(times[1:])), torch.cuda.max_memory_allocated(device) / 2 ** 30
    finally:
        net = x = out = None
        trim_memory()
        torch.cuda.empty_cache()


def autotune_dcn(device, use_amp, per_gpu_batch):
    """Pick the fastest NUMERICALLY EQUIVALENT deformable-conv implementation on this machine. Any problem ->
    falls back to torchvision (the v4.1 behaviour). Weights / checkpoints are unaffected either way."""
    DCN.backend, DCN.checkpoint = "torchvision", False
    mode = CFG.DCN_BACKEND
    if mode == "torchvision":
        log("DCN backend: torchvision (forced by CFG.DCN_BACKEND)")
        return
    try:
        err, errs = _dcn_selftest(device)
        ok = err < 2e-3
        log(f"DCN self-test (gridsample vs torchvision, fp32, fwd + 4 grads): max rel. error {err:.1e} -> "
            f"{'OK' if ok else 'MISMATCH'}")
        if ok and use_amp:
            err2, _ = _dcn_selftest(device, amp=True)
            ok = err2 < 5e-2
            log(f"DCN self-test under fp16 autocast: max rel. error {err2:.1e} -> {'OK' if ok else 'MISMATCH'}")
    except Exception as e:
        ok = False
        log(f"DCN self-test crashed: {str(e)[:200]}")
    if not ok:
        log("-> keeping torchvision deformable conv (old behaviour, slow but proven).")
        return
    try:
        ferr = _fuse_selftest(device)
        if ferr > 1e-3:
            CFG.FUSE_OFFSET_MASK = False
            log(f"fused offset/mask conv MISMATCH ({ferr:.1e}) -> disabled")
        else:
            log(f"fused offset/mask conv self-test OK (max abs diff {ferr:.1e})")
    except Exception as e:
        CFG.FUSE_OFFSET_MASK = False
        log(f"fused offset/mask self-test crashed ({str(e)[:120]}) -> disabled")

    key = (f"{torch.cuda.get_device_name(device)}|b{per_gpu_batch}|p{CFG.PATCH}|amp{int(use_amp)}|"
           f"{torch.__version__}|{CFG.DCN_BACKEND}|{CFG.DCN_CHECKPOINT}|{int(CFG.FUSE_OFFSET_MASK)}")
    cache_path = os.path.join(CFG.WORK_DIR, "dcn_choice.json")
    try:
        with open(cache_path) as f:
            cached = json.load(f)
        if cached.get("key") == key:
            DCN.backend, DCN.checkpoint = cached["backend"], bool(cached["checkpoint"])
            log(f"DCN backend (cached from the first start of this session): {DCN.backend}"
                f"{' + checkpointing' if DCN.checkpoint else ''}")
            return
    except Exception:
        pass
    ck_opts = {"auto": [False, True], True: [True], False: [False]}.get(CFG.DCN_CHECKPOINT, [False, True])
    cands = [] if mode == "gridsample" else [("torchvision", False)]
    cands += [("gridsample", c) for c in ck_opts]
    total_gb = torch.cuda.get_device_properties(device).total_memory / 2 ** 30
    limit_gb = (0.70 if CFG.STAGE == 3 else 0.80) * total_gb      # head-room for EMA/optimizer/VGG/D/gather
    log(f"DCN autotune: per-GPU batch {per_gpu_batch}, {CFG.PATCH}px, AMP={use_amp} (takes ~1 min) ...")
    results = []
    for name, ck in cands:
        DCN.backend, DCN.checkpoint = name, ck
        label = name + ("+ckpt" if ck else "")
        try:
            t, peak = _bench_dcn(device, use_amp, per_gpu_batch, 3)
            if name != "torchvision" and peak > limit_gb:
                log(f"   {label:18s} {t:6.2f} s/step  peak {peak:.1f} GB  -> too much VRAM (limit {limit_gb:.1f} GB), skipped")
                continue
            log(f"   {label:18s} {t:6.2f} s/step  peak {peak:.1f} GB")
            results.append((t, name, ck))
        except torch.cuda.OutOfMemoryError:
            log(f"   {label:18s} CUDA OOM, skipped")
        except Exception as e:
            log(f"   {label:18s} failed: {str(e)[:150]}")
        trim_memory(); torch.cuda.empty_cache()
    if not results:
        DCN.backend, DCN.checkpoint = "torchvision", False
        log("DCN autotune found no working candidate -> torchvision.")
        return
    t, name, ck = min(results)
    DCN.backend, DCN.checkpoint = name, ck
    base = next((r[0] for r in results if r[1] == "torchvision"), None)
    log(f"DCN backend chosen: {name}{' + checkpointing' if ck else ''}"
        + (f"  ({base / t:.1f}x faster than torchvision)" if base and name != "torchvision" else ""))
    try:
        with open(cache_path, "w") as f:
            json.dump({"key": key, "backend": name, "checkpoint": bool(ck)}, f)
    except Exception:
        pass


## Discriminator, perceptual loss, losses, EMA, schedules

In [10]:
SN = nn.utils.spectral_norm


class UNetDiscriminatorSN(nn.Module):
    """Real-ESRGAN's U-Net discriminator with spectral norm (per-pixel real/fake map)."""
    def __init__(self, nf=64):
        super().__init__()
        self.conv0 = nn.Conv2d(3, nf, 3, 1, 1)
        self.conv1 = SN(nn.Conv2d(nf, nf * 2, 4, 2, 1, bias=False))
        self.conv2 = SN(nn.Conv2d(nf * 2, nf * 4, 4, 2, 1, bias=False))
        self.conv3 = SN(nn.Conv2d(nf * 4, nf * 8, 4, 2, 1, bias=False))
        self.conv4 = SN(nn.Conv2d(nf * 8, nf * 4, 3, 1, 1, bias=False))
        self.conv5 = SN(nn.Conv2d(nf * 4, nf * 2, 3, 1, 1, bias=False))
        self.conv6 = SN(nn.Conv2d(nf * 2, nf, 3, 1, 1, bias=False))
        self.conv7 = SN(nn.Conv2d(nf, nf, 3, 1, 1, bias=False))
        self.conv8 = SN(nn.Conv2d(nf, nf, 3, 1, 1, bias=False))
        self.conv9 = nn.Conv2d(nf, 1, 3, 1, 1)

    def forward(self, x):
        lr = lambda t: F.leaky_relu(t, 0.2)
        up = lambda t: F.interpolate(t, scale_factor=2, mode="bilinear", align_corners=False)
        x0 = lr(self.conv0(x))
        x1 = lr(self.conv1(x0))
        x2 = lr(self.conv2(x1))
        x3 = lr(self.conv3(x2))
        x4 = lr(self.conv4(up(x3))) + x2
        x5 = lr(self.conv5(up(x4))) + x1
        x6 = lr(self.conv6(up(x5))) + x0
        return self.conv9(lr(self.conv8(lr(self.conv7(x6)))))


class VGGPerceptual(nn.Module):
    """VGG19 feature L1 on conv1_2 .. conv5_4 (pre-activation), weights as in Real-ESRGAN."""
    LAYERS = {2: 0.1, 7: 0.1, 16: 1.0, 25: 1.0, 34: 1.0}

    def __init__(self, weights_path=""):
        super().__init__()
        if weights_path and os.path.exists(weights_path):
            vgg = torchvision.models.vgg19(weights=None)
            vgg.load_state_dict(torch.load(weights_path, map_location="cpu"))
        else:
            vgg = torchvision.models.vgg19(weights=torchvision.models.VGG19_Weights.IMAGENET1K_V1)
        self.features = vgg.features[:35].eval()
        for m in self.features:
            if isinstance(m, nn.ReLU):
                m.inplace = False
        for p in self.features.parameters():
            p.requires_grad_(False)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, pred, target):
        x = (torch.cat([pred, target.detach()], 0).float() + 1) / 2
        h, loss = (x - self.mean) / self.std, 0.0
        for i, layer in enumerate(self.features):
            h = layer(h)
            if i in self.LAYERS:
                pf, tf = h.float().chunk(2)
                loss = loss + self.LAYERS[i] * F.l1_loss(pf, tf.detach())
        return loss


class PixelLoss(nn.Module):
    """Charbonnier + FFT + gradient (edge) loss, always fp32."""
    def __init__(self, fft_w, grad_w, eps=1e-6):
        super().__init__()
        self.fft_w, self.grad_w, self.eps = fft_w, grad_w, eps

    def forward(self, pred, target):
        pred, target = pred.float(), target.float()
        char = torch.sqrt((pred - target) ** 2 + self.eps).mean()
        total = char
        if self.fft_w > 0:
            total = total + self.fft_w * (torch.fft.rfft2(pred, norm="ortho") - torch.fft.rfft2(target, norm="ortho")).abs().mean()
        if self.grad_w > 0:
            gx = (pred[..., :, 1:] - pred[..., :, :-1]) - (target[..., :, 1:] - target[..., :, :-1])
            gy = (pred[..., 1:, :] - pred[..., :-1, :]) - (target[..., 1:, :] - target[..., :-1, :])
            total = total + self.grad_w * (gx.abs().mean() + gy.abs().mean())
        return total, char.detach()


def l2sp_penalty(core, anchor):
    """L2-SP: pull weights back toward the previous stage's solution (anti catastrophic forgetting)."""
    s = 0.0
    for n, p in core.named_parameters():
        s = s + ((p - anchor[n]) ** 2).sum()
    return s


class EMA:
    def __init__(self, model, decay):
        self.decay = decay
        self.module = copy.deepcopy(model).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model, step):
        d = min(self.decay, (1 + step) / (10 + step))
        ema_p = list(self.module.parameters())                           # parameters() de-dups tied weights
        torch._foreach_mul_(ema_p, d)
        torch._foreach_add_(ema_p, [p.detach() for p in model.parameters()], alpha=1 - d)


def _cycle_lengths(total, t0_steps, t_mult):
    """Cosine-restart cycle lengths (in steps) that add up to EXACTLY `total`: whole cycles while they fit; a short
    remainder is merged into the last cycle, a long one becomes its own final cycle."""
    lens, used, L = [], 0, max(1, int(t0_steps))
    while used + L <= total:
        lens.append(L)
        used += L
        L = max(1, int(round(L * t_mult)))
    rem = total - used
    if rem > 0:
        if lens and rem < 0.5 * lens[-1]:
            lens[-1] += rem
        else:
            lens.append(rem)
    return lens or [max(1, total)]


def lr_cycles_epochs(spe, scfg):
    total = max(1, scfg["epochs"] * spe - scfg["warmup"])
    return [L / spe for L in _cycle_lengths(total, scfg["t0"] * spe, scfg["t_mult"])]


def lr_at(stage_step, spe, scfg, key="lr"):
    """Linear warmup, then cosine annealing with warm restarts (each restart peak decays). The cycles are fitted to the
    stage length, so the LR is at its minimum (1e-6) on the last step instead of being cut off mid-cycle."""
    peak0, lo = scfg[key], 1e-6
    if stage_step < scfg["warmup"]:
        return peak0 * (stage_step + 1) / scfg["warmup"]
    total = max(1, scfg["epochs"] * spe - scfg["warmup"])
    s, cyc = stage_step - scfg["warmup"], 0
    for L in _cycle_lengths(total, scfg["t0"] * spe, scfg["t_mult"]):
        if s < L:
            break
        s -= L
        cyc += 1
    else:
        return lo                                   # past the planned end
    peak = peak0 * (scfg["rdecay"] ** cyc)
    return lo + (max(peak, lo) - lo) * 0.5 * (1 + math.cos(math.pi * s / L))


def severity_at(epoch, scfg):
    return min(1.0, scfg["sev0"] + (1.0 - scfg["sev0"]) * epoch / max(1, scfg["ramp"]))

## Checkpointing

In [11]:
def finite_params(model):
    return all(torch.isfinite(p).all().item() for p in model.parameters())


def atomic_save(obj, path):
    tmp = path + ".tmp"
    for attempt in (0, 1):
        try:
            make_room(os.path.getsize(path) if os.path.exists(path) else 64 * 2 ** 20, os.path.basename(path))
            torch.save(obj, tmp)
            os.replace(tmp, path)
            return True
        except (OSError, RuntimeError) as e:          # torch raises RuntimeError when the disk is full
            log(f"saving {os.path.basename(path)} failed ({str(e)[:100]})"
                + (" - freeing space and retrying" if attempt == 0 else ""))
            if os.path.exists(tmp):
                _rm(tmp, "partial file")
            cleanup_workdir(2)
    log(f"!!! could not save {os.path.basename(path)} - training continues without it")
    return False


def _strip(sd):
    return {(k[7:] if k.startswith("module.") else k): v for k, v in sd.items()}


def save_ckpt(C, epoch, note=""):
    if not (finite_params(C.core) and finite_params(C.ema.module)):
        log("REJECTED checkpoint: NaN/Inf in weights - keeping previous one.")
        return False
    latest, prev = os.path.join(CFG.WORK_DIR, "ckpt_latest.pth"), os.path.join(CFG.WORK_DIR, "ckpt_prev.pth")
    tmp = latest + ".tmp"
    obj = {"stage": CFG.STAGE, "epoch": int(epoch), "epoch_step": int(getattr(C, "epoch_step", 0)),
           "stage_step": int(C.stage_step), "total_step": int(C.total_step),
           "best_score": float(C.best_score), "base_div": float(C.base_div),
           "model": C.core.state_dict(), "ema": C.ema.module.state_dict(),
           "opt": C.opt.state_dict(), "scaler": C.scaler.state_dict()}
    if C.anchor is not None:
        obj["anchor"] = {k: v.cpu() for k, v in C.anchor.items()}
    if C.D is not None:
        obj.update({"D": C.D.state_dict(), "optD": C.optD.state_dict(), "scalerD": C.scalerD.state_dict()})
    for attempt in (0, 1):
        try:
            make_room(os.path.getsize(latest) if os.path.exists(latest) else 400 * 2 ** 20, "checkpoint")
            torch.save(obj, tmp)
            if os.path.exists(latest):
                os.replace(latest, prev)
            os.replace(tmp, latest)
            log(f"CHECKPOINT saved (stage {CFG.STAGE}, epoch {epoch}, step {C.total_step}) {note}")
            return True
        except (OSError, RuntimeError) as e:
            log(f"checkpoint save failed ({str(e)[:100]})" + (" - freeing space and retrying" if attempt == 0 else ""))
            if os.path.exists(tmp):
                _rm(tmp, "partial file")
            cleanup_workdir(2)
    log("!!! CHECKPOINT NOT SAVED (no space) - the previous one is still there; training continues")
    return False


def find_checkpoint():
    cands = [os.path.join(CFG.WORK_DIR, n) for n in ("ckpt_latest.pth", "ckpt_prev.pth")]
    for root, _, files in os.walk(CFG.INPUT_DIR):
        for f in files:
            if f in ("ckpt_latest.pth", "ckpt_prev.pth"):
                cands.append(os.path.join(root, f))
    best = None
    for p in cands:
        if not os.path.exists(p):
            continue
        try:
            try:
                ck = torch.load(p, map_location="cpu", mmap=True)
            except Exception:
                ck = torch.load(p, map_location="cpu")
            if "model" in ck and (best is None or ck.get("total_step", 0) > best[0]):
                best = (ck.get("total_step", 0), p)
            del ck
        except Exception as e:
            log(f"unreadable checkpoint {p}: {str(e)[:120]}")
    if best is None:
        return None, None
    return torch.load(best[1], map_location="cpu"), best[1]


def find_legacy_weights():
    for root, _, files in os.walk(CFG.INPUT_DIR):
        for f in files:
            if f.lower().endswith((".pth", ".pt")) and "lego" in f.lower() and not f.startswith("ckpt_"):
                yield os.path.join(root, f)

## Validation

In [12]:
_LAP = torch.tensor([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=torch.float32).view(1, 1, 3, 3)


def lap_var(x01):
    g = x01.mean(1, keepdim=True)
    return F.conv2d(g, _LAP.to(g.device)).flatten(1).var(dim=1)


def psnr01(a, b):
    mse = ((a - b) ** 2).flatten(1).mean(1).clamp_min(1e-10)
    return 10 * torch.log10(1.0 / mse)


@torch.no_grad()
def validate(net, val_set, device, tag, use_amp):
    net.eval()
    deg_all, clean_all = val_set
    p_in, p_out, s_in, s_out, s_gt, first = [], [], [], [], [], None
    for i in range(0, deg_all.shape[0], 4):
        deg = to_model_range(deg_all[i:i + 4].to(device))
        clean = to_model_range(clean_all[i:i + 4].to(device))
        with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
            out = net(deg)
        out = out.float().clamp(-1, 1)
        d01, o01, c01 = (deg + 1) / 2, (out + 1) / 2, (clean + 1) / 2
        p_in.append(psnr01(d01, c01)); p_out.append(psnr01(o01, c01))
        s_in.append(lap_var(d01)); s_out.append(lap_var(o01)); s_gt.append(lap_var(c01))
        if first is None:
            first = (d01.cpu(), o01.cpu(), c01.cpu())
    gt = torch.cat(s_gt).mean().item() + 1e-12
    m = dict(psnr_in=torch.cat(p_in).mean().item(), psnr_out=torch.cat(p_out).mean().item(),
             sharp_in=torch.cat(s_in).mean().item() / gt, sharp_out=torch.cat(s_out).mean().item() / gt)
    m["gain"] = m["psnr_out"] - m["psnr_in"]
    os.makedirs(SAMPLES_DIR, exist_ok=True)
    rows = []
    for i in range(min(4, first[0].shape[0])):
        trip = [(t[i].permute(1, 2, 0).numpy() * 255).round().clip(0, 255).astype(np.uint8) for t in first]
        rows.append(np.concatenate(trip, axis=1))          # input | restored | ground truth
    cv2.imwrite(os.path.join(SAMPLES_DIR, f"{tag}.png"), cv2.cvtColor(np.concatenate(rows, 0), cv2.COLOR_RGB2BGR))
    return m


def fmt(m):
    return (f"PSNR in {m['psnr_in']:.2f} -> out {m['psnr_out']:.2f} ({m['gain']:+.2f} dB) | "
            f"sharpness in {m['sharp_in']:.2f} -> out {m['sharp_out']:.2f} (1.0 = like ground truth)")


class Ctx:   # mutable training state shared with the checkpoint helper
    pass

## Diagnostics: step-wait timer + startup pipeline profile


In [13]:
def _sync_all():
    torch.cuda.synchronize(DIST.local)          # only THIS process's GPU (never create a context on the other one)


def _timed(it, acc):
    """yields the items of `it`; adds the seconds spent WAITING for each item to acc[0] (log window) and acc[1] (total)."""
    it = iter(it)
    while True:
        t0 = time.time()
        try:
            item = next(it)
        except StopIteration:
            return
        w = time.time() - t0
        acc[0] += w
        acc[1] += w
        yield item


_THROTTLE = {0x1: "idle", 0x2: "app-clocks", 0x4: "POWER-CAP", 0x8: "HW-slowdown", 0x20: "SW-thermal",
             0x40: "HW-THERMAL", 0x80: "POWER-BRAKE"}


def _throttle_names(hexstr):
    try:
        v = int(hexstr, 16)
    except Exception:
        return hexstr
    names = [n for bit, n in _THROTTLE.items() if v & bit]
    return "+".join(names) if names else "none"


def gpu_line():
    """Per-GPU utilisation (avg of 4 samples), SM clock vs max, power, temperature, throttle reason, allocator state."""
    import subprocess
    fields = "utilization.gpu,clocks.sm,clocks.max.sm,power.draw,power.limit,temperature.gpu,clocks_throttle_reasons.active"
    rows, util = None, None
    try:
        for k in range(4):
            r = subprocess.run(["nvidia-smi", f"--query-gpu={fields}", "--format=csv,noheader,nounits"],
                               capture_output=True, text=True, timeout=10)
            lines = [l for l in r.stdout.strip().splitlines() if l.strip()]
            if r.returncode != 0 or not lines:
                raise RuntimeError(r.stderr[:80])
            rows = [[x.strip() for x in l.split(",")] for l in lines]
            u = [float(row[0]) for row in rows]
            util = u if util is None else [a + b for a, b in zip(util, u)]
            time.sleep(0.2)
        util = [x / 4 for x in util]
    except Exception as e:
        return f"GPU: nvidia-smi unavailable ({str(e)[:60]})"
    out = []
    for g, row in enumerate(rows):
        try:
            clk, clk_max, p, pl, t, thr = row[1], row[2], float(row[3]), float(row[4]), row[5], _throttle_names(row[6])
            s = f"GPU{g} util {util[g]:.0f}% | SM {clk}/{clk_max} MHz | {p:.0f}/{pl:.0f} W | {t} C | throttle: {thr}"
        except Exception:
            s = f"GPU{g} util {util[g]:.0f}%"
        try:
            if g != DIST.local:
                raise RuntimeError("memory stats only for this process's GPU")
            st = torch.cuda.memory_stats(g)
            s += (f" | VRAM peak {torch.cuda.max_memory_allocated(g) / 2 ** 30:.1f} GB, "
                  f"reserved {torch.cuda.memory_reserved(g) / 2 ** 30:.1f} GB, alloc-retries {st.get('num_alloc_retries', 0)}")
        except Exception:
            pass
        out.append(s)
    return "\n".join(out)


def report_phases(phase, n, wait_total):
    comp = sum(phase.values()) / n
    parts = " | ".join(f"{k} {v / n:.2f}s" for k, v in phase.items())
    wait = wait_total / max(1, n - 1)
    log(f"STEP PHASES (average of {n} synchronised steps): data wait {wait:.2f}s | {parts} | compute total {comp:.2f}s")
    if wait > 0.5 * (comp + wait):
        log("-> the GPUs mostly WAIT FOR DATA (CPU pipeline is the bottleneck).")
    else:
        worst = max(phase, key=phase.get)
        log(f"-> data is not the problem; the biggest share is '{worst}' ({100 * phase[worst] / sum(phase.values()):.0f}% of the step).")


def profile_pipeline(stage, train_paths, scfg, core, device, use_amp, spe):
    """~30 s, first start only, rank 0. (a) CPU cost of one decode / one degradation -> can the CPUs feed the GPUs?
    (b) ONE-GPU step time at the per-GPU batch = what a DDP step should cost. Compare it with the real 'it/s' lines."""
    B, P = DIST.bpr, CFG.PATCH
    imgs = []
    t0 = time.time()
    for p in random.sample(list(train_paths), min(4, len(train_paths))):
        im = read_rgb(p, P)
        if im is not None:
            imgs.append(im)
    t_dec = (time.time() - t0) / max(1, len(imgs))
    t0, k = time.time(), 0
    for _ in range(3):
        for im in imgs:
            degrade(random_crop_aug(im, P), 0.6, scfg["second_p"])
            k += 1
    t_deg = (time.time() - t0) / max(1, k)
    per_batch = (B / CFG.CROPS_PER_IMG) * t_dec + B * t_deg          # CPU-seconds to make one per-GPU DIV2K batch

    net = opt = sc = x = y = None
    t_one = peak = None
    try:
        net = copy.deepcopy(core).to(device).train()
        opt = torch.optim.AdamW(net.parameters(), lr=1e-7)
        sc = torch.amp.GradScaler("cuda", enabled=use_amp)
        x = torch.rand(B, 3, P, P, device=device) * 2 - 1
        y = torch.rand(B, 3, P, P, device=device) * 2 - 1

        def step():
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                out = net(x)
            loss = (out.float() - y).abs().mean()
            sc.scale(loss).backward()
            sc.unscale_(opt)
            nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            sc.step(opt)
            sc.update()

        torch.cuda.reset_peak_memory_stats(device)
        for _ in range(2):
            step()
        _sync_all()
        t0 = time.time()
        for _ in range(4):
            step()
        _sync_all()
        t_one = (time.time() - t0) / 4
        peak = torch.cuda.max_memory_allocated(device) / 2 ** 30
    finally:
        net = opt = sc = x = y = None
        trim_memory()
        torch.cuda.empty_cache()

    cpu_rate = CFG.NUM_WORKERS / per_batch
    log(f"PROFILE | CPU: decode {t_dec:.2f} s/image, degrade {t_deg * 1000:.0f} ms/crop -> a DIV2K GPU process can make "
        f"~{cpu_rate:.2f} batches/s with {CFG.NUM_WORKERS} workers")
    log(f"PROFILE | GPU: ONE GPU, batch {B}: {t_one:.2f} s/step (peak VRAM {peak:.1f} GB)")
    exp = t_one * (1.06 if DIST.on else 1.0)
    log(f"PROFILE | expected speed with {DIST.world} GPU process(es): ~{exp:.2f} s/step = {1 / exp:.2f} it/s "
        f"-> {spe} steps (one epoch) ~ {spe * exp / 60:.0f} min. If the real log shows far less, read 'STEP PHASES'.")
    if cpu_rate < 1.2 / exp:
        log(f"VERDICT: DIV2K decoding would be the bottleneck -> raise CFG.NUM_WORKERS (now {CFG.NUM_WORKERS}).")
    else:
        log("VERDICT: the CPUs can feed the GPUs; speed is set by GPU compute.")


## Main

In [14]:
def resolve_layout(stage):
    m = CFG.DATA_LAYOUT
    if m == "auto":
        m = {1: "split", 2: "split", 3: "mixed"}[stage]
    if m not in ("div2k", "split", "mixed"):
        raise ValueError(f"CFG.DATA_LAYOUT must be 'auto', 'div2k', 'split' or 'mixed' - got {m!r}")
    return m


def source_for_rank(layout, scfg, rank, world):
    """-> (kind, scfg of this rank).  kind: 'div2k' (DataLoader on DIV2K) | 'stream' (COCO+Flickr only) |
    'mixed' (COCO+Flickr plus scfg['replay'] share of DIV2K)."""
    if layout == "div2k":
        return "div2k", scfg
    if layout == "split":
        if world == 1:
            return "mixed", dict(scfg, replay=0.5)
        return ("div2k", scfg) if rank % 2 == 0 else ("stream", dict(scfg, replay=0.0))
    return "mixed", scfg


def describe_source(kind, rscfg):
    return {"div2k": "DIV2K", "stream": "COCO+Flickr stream",
            "mixed": f"COCO+Flickr stream + {rscfg['replay']:.0%} DIV2K replay"}[kind]


def main():
    os.makedirs(CFG.WORK_DIR, exist_ok=True)
    stage, scfg = CFG.STAGE, STAGES[CFG.STAGE]
    rank, world = DIST.rank, DIST.world
    device = torch.device("cuda", DIST.local)
    torch.cuda.set_device(device)
    use_amp = True
    torch.backends.cudnn.benchmark = True
    torch.set_num_threads(1)                    # the main process only launches kernels; spare the cores for the data workers
    seed_everything(DIST.attempt)
    DIST.bpr = max(1, min(CFG.MAX_BATCH_PER_GPU, -(-CFG.BATCH_PATCHES // world)))
    glob_b = DIST.bpr * world
    log(f"=== LEGO v5.0 | STAGE {stage} | "
        + (f"DDP x{world} ({DIST.backend})" if DIST.on else "single GPU")
        + f" | batch {DIST.bpr} per GPU = {glob_b} global | AMP={use_amp} ===")
    if glob_b != CFG.BATCH_PATCHES:
        log(f"!!! global batch is {glob_b} (not CFG.BATCH_PATCHES={CFG.BATCH_PATCHES}): one T4 holds at most "
            f"{CFG.MAX_BATCH_PER_GPU} patches. Steps/epoch and the LR schedule are unchanged, each step sees fewer patches.")

    # ---- data lists (every rank) + preflight (rank 0)
    info = scan_input(CFG.INPUT_DIR)
    if is_main():
        preflight(info)                  # raises with a clear reason if the run cannot work
    elif not info["images"]:
        raise RuntimeError("no training images visible")

    # ---- fastest equivalent deformable-conv: tuned on rank 0, shared with the others
    if is_main():
        autotune_dcn(device, use_amp, DIST.bpr)
    choice = broadcast_obj(dict(backend=DCN.backend, checkpoint=DCN.checkpoint, fuse=CFG.FUSE_OFFSET_MASK)
                           if is_main() else None)
    DCN.backend, DCN.checkpoint, CFG.FUSE_OFFSET_MASK = choice["backend"], choice["checkpoint"], choice["fuse"]

    paths = sorted(info["images"])
    random.Random(0).shuffle(paths)
    n_val = max(1, min(CFG.VAL_MAX_IMAGES, int(len(paths) * CFG.VAL_FRACTION)))
    val_paths, train_paths = paths[:n_val], paths[n_val:]
    log(f"DIV2K-style images: {len(train_paths)} train / {len(val_paths)} val (held out for all stages)")

    # ---- validation sets (rank 0 only) built BEFORE any worker/feeder thread starts
    val_div = val_str = None
    if is_main():
        val_clean = []
        for p in val_paths:
            im = read_rgb(p, CFG.PATCH)
            if im is not None:
                val_clean.append(center_crop(im, CFG.PATCH))
        val_div = make_val_set(val_clean, CFG.VAL_SEVERITY, 10_000)

    layout = resolve_layout(stage)
    if layout != "div2k":
        payload = None
        if is_main():
            try:
                crops = fetch_stream_val(CFG.PATCH, CFG.STREAM_VAL_IMAGES)
                val_str = make_val_set(crops, CFG.VAL_SEVERITY, 20_000)
                payload = {"ok": True, "hashes": list(STREAM_VAL_HASH)}
            except Exception as e:
                payload = {"ok": False, "err": str(e)[:400]}
        payload = broadcast_obj(payload)
        if payload["ok"]:
            STREAM_VAL_HASH.update(tuple(h) for h in payload["hashes"])     # every GPU must skip the validation images
            if is_main():
                log(f"stream validation set: {val_str[0].shape[0]} images")
        elif stage >= 2:
            raise RuntimeError(payload["err"])
        else:
            log("!!! COCO/Flickr stream unavailable (Internet OFF?) -> stage 1 trains on DIV2K only, on every GPU. "
                f"({payload['err'][:150]})")
            layout, val_str = "div2k", None
    kind, rscfg = source_for_rank(layout, scfg, rank, world)
    log(f"DATA LAYOUT '{layout}': " + " | ".join(
        f"GPU{r}: {describe_source(*source_for_rank(layout, scfg, r, world))}" for r in range(world)))
    severity = torch.tensor([severity_at(0, scfg)]).share_memory_()

    # ---- model
    C = Ctx()
    core = DeepSpatialRestorationNet(CFG.EMBED_DIM).to(device)
    C.core, C.ema = core, EMA(core, CFG.EMA_DECAY)
    decay, no_decay = [], []
    for n, p in core.named_parameters():
        (no_decay if p.ndim < 2 or "temperature" in n else decay).append(p)
    C.opt = torch.optim.AdamW([{"params": decay, "weight_decay": CFG.WEIGHT_DECAY},
                               {"params": no_decay, "weight_decay": 0.0}], lr=scfg["lr"])
    C.scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    C.D = C.optD = C.scalerD = C.anchor = None
    vgg = None
    if stage == 3:
        C.D = UNetDiscriminatorSN().to(device)
        C.optD = torch.optim.Adam(C.D.parameters(), lr=scfg["lr_d"], betas=(0.9, 0.99))
        C.scalerD = torch.amp.GradScaler("cuda", enabled=use_amp)
        try:
            vgg = VGGPerceptual(CFG.VGG_WEIGHTS_PATH).to(device).eval()
        except Exception as e:
            raise RuntimeError("Stage 3 needs VGG19 ImageNet weights: turn Internet ON, or set CFG.VGG_WEIGHTS_PATH "
                               f"to a vgg19 state_dict file. ({str(e)[:150]})")
    pix_loss = PixelLoss(scfg["fft_w"], scfg["grad_w"]).to(device)

    # ---- resume / hand-over (every rank loads the same file -> identical weights everywhere)
    C.total_step = C.stage_step = 0
    C.best_score, C.base_div = -1.0, None
    start_epoch, resume_skip, C.epoch_step = 0, 0, 0
    ck, ck_path = find_checkpoint()
    if ck is not None and ck["stage"] == stage:
        core.load_state_dict(_strip(ck["model"])); C.ema.module.load_state_dict(_strip(ck["ema"]))
        C.opt.load_state_dict(ck["opt"]); C.scaler.load_state_dict(ck["scaler"])
        if C.D is not None and "D" in ck:
            C.D.load_state_dict(ck["D"]); C.optD.load_state_dict(ck["optD"]); C.scalerD.load_state_dict(ck["scalerD"])
        if "anchor" in ck:
            C.anchor = {k: v.to(device) for k, v in ck["anchor"].items()}
        start_epoch, C.stage_step, C.total_step = ck["epoch"], ck["stage_step"], ck["total_step"]
        resume_skip = int(ck.get("epoch_step", 0))
        C.best_score, C.base_div = ck["best_score"], ck.get("base_div")
        log(f"FULL RESUME of stage {stage} from {ck_path}: epoch {start_epoch}, step {C.total_step}")
    elif ck is not None:
        core.load_state_dict(_strip(ck["ema"])); C.ema.module.load_state_dict(_strip(ck["ema"]))
        C.total_step = ck["total_step"]
        log(f"HAND-OVER: stage {ck['stage']} -> stage {stage} (EMA weights as init, fresh optimizer) from {ck_path}")
    else:
        loaded = None
        if CFG.WARM_START_OLD:
            for p in find_legacy_weights():
                try:
                    core.load_state_dict(_strip(torch.load(p, map_location="cpu")))
                    if finite_params(core):
                        loaded = p
                        break
                except Exception as e:
                    log(f"legacy weights {p} not usable: {str(e)[:100]}")
            if loaded:
                C.ema.module.load_state_dict(core.state_dict())
        log(f"WARM START (weights only) from {loaded}" if loaded else "STARTING FROM SCRATCH")
        if not loaded and scfg["l2sp"] > 0:
            scfg = dict(scfg, l2sp=0.0)
            rscfg = dict(rscfg, l2sp=0.0)
            log("No previous solution to anchor to -> L2-SP disabled for this run.")
    del ck
    trim_memory()
    C.ema.module.to(device)
    for stt in C.opt.state.values():
        for k, v in stt.items():
            if torch.is_tensor(v):
                stt[k] = v.to(device)
    if stage >= 2 and C.anchor is None:       # L2-SP anchor = weights we start this stage from
        C.anchor = {n: p.detach().clone() for n, p in core.named_parameters()}

    # ---- DDP wrappers (built AFTER the weights are loaded: DDP broadcasts rank 0's weights to the others)
    if DIST.on:
        model = DDP(core, device_ids=[DIST.local], output_device=DIST.local,
                    broadcast_buffers=False, find_unused_parameters=False)
        # D is wrapped for ITS OWN step only; the generator step calls the plain C.D (shared weights), because that step runs with
        # D's parameters frozen and DDP would otherwise wait for gradient hooks that never fire.
        D_ddp = DDP(C.D, device_ids=[DIST.local], output_device=DIST.local,
                    broadcast_buffers=False, find_unused_parameters=False) if C.D is not None else None
    else:
        model, D_ddp = core, C.D

    spe = scfg["spe"] or (len(train_paths) * CFG.CROPS_PER_IMG // CFG.BATCH_PATCHES)   # steps/epoch: independent of GPU count
    total_epochs = scfg["epochs"]

    # ---- baseline (also the reference for the forgetting check) - rank 0
    if is_main():
        m = validate(C.ema.module, val_div, device, f"s{stage}_baseline_div", use_amp)
        log(f"BASELINE DIV2K-val  {fmt(m)}")
        if C.base_div is None:
            C.base_div = m["psnr_out"]
        if val_str is not None:
            log(f"BASELINE stream-val {fmt(validate(C.ema.module, val_str, device, f's{stage}_baseline_stream', use_amp))}")
        if not os.path.exists(METRICS_CSV):
            with open(METRICS_CSV, "w") as f:
                f.write("stage,epoch,total_step,lr,train_charb,div_psnr_in,div_psnr_out,div_sharp_out,"
                        "str_psnr_out,str_sharp_out,ram_gb,minutes\n")
        if CFG.PROFILE_AT_START and os.environ.get("LEGO_RESTART", "0") == "0":
            try:
                profile_pipeline(stage, train_paths, scfg, core, device, use_amp, spe)
            except Exception as e:
                log(f"profile skipped: {str(e)[:200]}")
                trim_memory(); torch.cuda.empty_cache()
    if C.base_div is None:
        C.base_div = 0.0

    # ---- data source of THIS GPU
    feeder, train_dl, batch_gen = None, None, None
    if kind == "div2k":
        train_ds = Stage1Dataset(train_paths, CFG.PATCH, CFG.CROPS_PER_IMG, severity, rscfg["second_p"],
                                 length=spe * DIST.bpr)
        train_dl = DataLoader(train_ds, batch_size=DIST.bpr, shuffle=True, num_workers=CFG.NUM_WORKERS, drop_last=True,
                              pin_memory=CFG.PIN_MEMORY, persistent_workers=False, worker_init_fn=worker_init,
                              prefetch_factor=CFG.PREFETCH if CFG.NUM_WORKERS > 0 else None)
        assert len(train_dl) == spe, (len(train_dl), spe)
    else:
        feeder = MixedFeeder(train_paths, severity, rscfg)
        batch_gen = feeder.batches()
    log(f"stage {stage}: {total_epochs} epochs x {spe} steps | global batch {glob_b} patches ({DIST.bpr}/GPU) | "
        f"peak LR {scfg['lr']:.1e} | replay {rscfg['replay']:.0%} | L2-SP {scfg['l2sp']}")
    log("LR schedule (cosine cycle lengths in epochs, ends at 1e-6 on the last step): "
        + ", ".join(f"{e:.1f}" for e in lr_cycles_epochs(spe, scfg)))

    last_ckpt, oom_count, stop_reason, lr = time.time(), 0, None, scfg["lr"]
    steps_at_start, t_first = C.total_step, time.time()
    log("Starting training loop - waiting for the first batch (workers warming up)...")
    ones_like = lambda t: torch.ones_like(t, dtype=torch.float32)
    wait_acc = [0.0, 0.0]
    phase, lap_t, w_at, prof = {}, [0.0], [0.0], False
    is_restart = os.environ.get("LEGO_RESTART", "0") == "1"
    CTRL = {1: "time budget", 2: "memory"}

    def _lap(name):
        if prof:
            _sync_all()
            now = time.time()
            phase[name] = phase.get(name, 0.0) + now - lap_t[0]
            lap_t[0] = now
    try:
        for epoch in range(start_epoch, total_epochs):
            if sync_max(1 if (is_main() and hours() >= CFG.MAX_SESSION_HOURS) else 0):
                stop_reason = "time budget"
                break
            severity[0] = severity_at(epoch, scfg)
            model.train()
            n0 = resume_skip if (epoch == start_epoch and 0 < resume_skip < spe) else 0     # steps of this epoch already done
            t_ep, run_c, n_it, skipped, finished = time.time(), 0.0, n0, 0, True
            t_win = time.time()
            C.epoch_step = n_it
            log(f"=== STAGE {stage} EPOCH {epoch + 1}/{total_epochs} | severity <= {float(severity[0]):.2f} ===")
            batch_iter = _timed(itertools.islice(iter(train_dl) if train_dl is not None else batch_gen, spe - n0), wait_acc)
            for deg_u8, clean_u8 in batch_iter:
                # ---- one agreed decision for all GPUs every CTRL_EVERY steps (stop / restart); only rank 0 looks at clock + RAM
                if n_it % CFG.CTRL_EVERY == 0:
                    code = 0
                    if is_main():
                        if hours() >= CFG.MAX_SESSION_HOURS:
                            code = 1
                        elif n_it % 25 == 0:
                            rs, rmsg = ram_state()
                            if rs == "stop":
                                log(f"MEMORY GUARD: {rmsg} - saving a checkpoint and "
                                    + ("restarting the training process." if IN_SUPERVISOR else "stopping before the OOM killer."))
                                code = 2
                    code = int(sync_max(code))
                    if code:
                        finished, stop_reason = False, CTRL[code]
                        break
                if n_it % 50 == 0:
                    trim_memory()
                k_ = n_it - n0
                prof = CFG.PHASE_PROFILE and not is_restart and epoch == start_epoch and \
                    CFG.PHASE_START <= k_ < CFG.PHASE_START + CFG.PHASE_STEPS
                if prof:
                    if k_ == CFG.PHASE_START:
                        w_at[0] = wait_acc[1]
                    _sync_all()
                    lap_t[0] = time.time()
                deg = to_model_range(deg_u8.to(device, non_blocking=True))
                clean = to_model_range(clean_u8.to(device, non_blocking=True))
                _lap("host->GPU")
                lr = lr_at(C.stage_step, spe, scfg)
                for g in C.opt.param_groups:
                    g["lr"] = lr
                if stage == 3:
                    for g in C.optD.param_groups:
                        g["lr"] = lr_at(C.stage_step, spe, scfg, "lr_d")
                C.opt.zero_grad(set_to_none=True)
                try:
                    if stage == 3:
                        for p in C.D.parameters():
                            p.requires_grad_(False)
                    with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                        restored = model(deg)
                    pix, char = pix_loss(restored, clean)
                    loss = pix
                    if stage == 3:
                        with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                            perc = vgg(restored, clean)
                            fake_g = C.D(restored)                  # plain D (frozen) - see DDP note above
                        adv = F.binary_cross_entropy_with_logits(fake_g.float(), ones_like(fake_g))
                        loss = scfg["pix_w"] * pix + scfg["perc_w"] * perc + scfg["gan_w"] * adv
                    if scfg["l2sp"] > 0:
                        loss = loss + scfg["l2sp"] * l2sp_penalty(core, C.anchor)
                    if stage == 3:
                        for p in C.D.parameters():
                            p.requires_grad_(True)
                    _lap("forward+loss")
                    if sync_min(1.0 if bool(torch.isfinite(loss)) else 0.0) < 1.0:    # ALL ranks skip together (else the all-reduce hangs)
                        skipped += 1
                        continue
                    C.scaler.scale(loss).backward()                 # DDP: gradient all-reduce overlaps with this
                    _lap("backward (+all-reduce)")
                    C.scaler.unscale_(C.opt)
                    nn.utils.clip_grad_norm_(core.parameters(), CFG.GRAD_CLIP)
                    C.scaler.step(C.opt)
                    C.scaler.update()
                    _lap("optimizer")
                    if stage == 3:                                  # discriminator step (real + fake in ONE forward: no BatchNorm -> identical maths)
                        C.optD.zero_grad(set_to_none=True)
                        with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                            both = D_ddp(torch.cat([clean, restored.detach().to(clean.dtype)], 0))
                        real_l, fake_l = both.chunk(2)
                        d_loss = 0.5 * (F.binary_cross_entropy_with_logits(real_l.float(), ones_like(real_l)) +
                                        F.binary_cross_entropy_with_logits(fake_l.float(), torch.zeros_like(fake_l, dtype=torch.float32)))
                        C.scalerD.scale(d_loss).backward()
                        C.scalerD.step(C.optD)
                        C.scalerD.update()
                        _lap("D step")
                except torch.cuda.OutOfMemoryError:
                    if DIST.on:      # one GPU alone cannot recover: the other one would wait in the all-reduce
                        log("!!! CUDA OOM - all GPUs stop. Lower CFG.BATCH_PATCHES (e.g. 6 or 4) and run again (it resumes).",
                            all_ranks=True)
                        raise
                    oom_count += 1
                    log(f"CUDA OOM #{oom_count} - skipping batch (lower CFG.BATCH_PATCHES if this repeats)")
                    restored = loss = None
                    if stage == 3:
                        for p in C.D.parameters():
                            p.requires_grad_(True)
                    C.opt.zero_grad(set_to_none=True)
                    trim_memory(); torch.cuda.empty_cache()
                    if oom_count > 25:
                        raise
                    continue
                C.ema.update(core, C.total_step)
                _lap("EMA")
                C.stage_step += 1; C.total_step += 1; n_it += 1; C.epoch_step = n_it
                if prof and (n_it - n0) == CFG.PHASE_START + CFG.PHASE_STEPS:
                    report_phases(phase, CFG.PHASE_STEPS, wait_acc[1] - w_at[0])
                    prof = False
                if C.total_step == steps_at_start + 1:
                    log(f"FIRST STEP DONE after {time.time() - t_first:.0f}s - training is running. loss {char.item():.4f}")
                run_c = run_c + char                  # stays on the GPU (no per-step sync)
                if n_it % CFG.LOG_EVERY_STEPS == 0:
                    losses = gather_scalar(float(run_c) / max(1, n_it - n0))     # collective: every rank
                    if is_main():
                        extra = ""
                        if scfg["l2sp"] > 0:
                            extra += f" | l2sp {float(l2sp_penalty(core, C.anchor)):.2f}"
                        if stage == 3:
                            extra += f" | D {d_loss.item():.3f} adv {adv.item():.3f} perc {float(perc):.3f}"
                        now = time.time()
                        rate = CFG.LOG_EVERY_STEPS / max(now - t_win, 1e-6)
                        t_win = now
                        eta_h = ((total_epochs - epoch - 1) * spe + (spe - n_it)) / max(rate, 1e-6) / 3600
                        wait_s, wait_acc[0] = wait_acc[0] / CFG.LOG_EVERY_STEPS, 0.0
                        loss_s = f"{sum(losses) / len(losses):.4f}"
                        if world > 1:
                            loss_s += " (" + " / ".join(f"GPU{i} {v:.4f}" for i, v in enumerate(losses)) + ")"
                        log(f"s{stage} ep{epoch + 1} it{n_it}/{spe} | charb {loss_s} | lr {lr:.2e} | "
                            f"{rate:.2f} it/s ({1 / max(rate, 1e-6):.1f} s/step, data wait {wait_s:.1f} s) | "
                            f"stage ETA {eta_h:.1f}h | skipped {skipped}{extra}")
                    else:
                        t_win, wait_acc[0] = time.time(), 0.0
                if is_main() and n_it % (CFG.LOG_EVERY_STEPS * 4) == 0:
                    rmsg = ram_state()[1]
                    log(rmsg if "children" in rmsg else rmsg + " | " + ram_breakdown())
                    log(gpu_line())
                if is_main() and time.time() - last_ckpt > CFG.CKPT_EVERY_MIN * 60:
                    save_ckpt(C, epoch, "(timed)")
                    last_ckpt = time.time()

            batch_iter = None                                     # release DataLoader workers now
            if not finished:
                if is_main():
                    save_ckpt(C, epoch, f"({stop_reason})")
                break

            # ---- end of epoch: rank 0 validates (+ forgetting check) and saves; the others wait in the next collective
            ep_losses = gather_scalar(float(run_c) / max(1, n_it - n0))
            ctrl = 0
            if is_main():
                tag = f"s{stage}_e{epoch + 1:03d}"
                tr_loss = sum(ep_losses) / len(ep_losses)
                m = validate(C.ema.module, val_div, device, tag + "_div", use_amp)
                mins = (time.time() - t_ep) / 60
                log(f"VAL epoch {epoch + 1} DIV2K : {fmt(m)} | train charb {tr_loss:.4f} | {mins:.1f} min")
                ms = None
                if val_str is not None:
                    ms = validate(C.ema.module, val_str, device, tag + "_stream", use_amp)
                    log(f"VAL epoch {epoch + 1} STREAM: {fmt(ms)}")
                    drop = C.base_div - m["psnr_out"]
                    log(f"FORGETTING CHECK: DIV2K-val PSNR is {-drop:+.2f} dB vs the start of this stage"
                        + ("  <-- WARNING: forgetting! raise l2sp or replay in STAGES" if drop > CFG.FORGET_WARN_DB and stage == 2 else ""))
                with open(METRICS_CSV, "a") as f:
                    f.write(f"{stage},{epoch + 1},{C.total_step},{lr:.3e},{tr_loss:.5f},{m['psnr_in']:.3f},"
                            f"{m['psnr_out']:.3f},{m['sharp_out']:.3f},{ms['psnr_out'] if ms else ''},"
                            f"{ms['sharp_out'] if ms else ''},{ram_usage()[0]:.1f},{mins:.1f}\n")
                if finite_params(C.ema.module):
                    atomic_save(C.ema.module.state_dict(), os.path.join(CFG.WORK_DIR, "LEGO_latest.pth"))
                    atomic_save(C.ema.module.state_dict(), os.path.join(CFG.WORK_DIR, f"LEGO_s{stage}_latest.pth"))
                    if stage == 3:      # PSNR is the wrong judge for GAN output: keep the last 3 epochs, pick by eye
                        atomic_save(C.ema.module.state_dict(), os.path.join(CFG.WORK_DIR, f"LEGO_s3_ep{epoch + 1:03d}.pth"))
                        old = os.path.join(CFG.WORK_DIR, f"LEGO_s3_ep{epoch - 2:03d}.pth")
                        if os.path.exists(old):
                            os.remove(old)
                    else:
                        score = m["psnr_out"] if ms is None else 0.5 * (m["psnr_out"] + ms["psnr_out"])
                        if score > C.best_score:
                            C.best_score = score
                            atomic_save(C.ema.module.state_dict(), os.path.join(CFG.WORK_DIR, f"LEGO_s{stage}_best.pth"))
                            log(f"NEW BEST (score {score:.2f} dB) -> LEGO_s{stage}_best.pth")
                C.epoch_step = 0
                save_ckpt(C, epoch + 1, "(epoch end)")
                trim_memory(); torch.cuda.empty_cache()
                used_, lim_ = ram_usage()
                if IN_SUPERVISOR and epoch + 1 < total_epochs and used_ / max(lim_, 1e-6) >= CFG.EPOCH_RESTART_FRAC:
                    log(f"RAM {used_:.1f}/{lim_:.1f} GB at the epoch end -> clean restart of the training process "
                        f"(exact resume, about a minute)")
                    ctrl = 2
            C.epoch_step = 0
            last_ckpt = time.time()
            if int(sync_max(ctrl)) == 2:                          # also the 'barrier' that releases the waiting GPUs
                stop_reason = "memory"
                break
            model.train()
            trim_memory()
        else:
            stop_reason = "stage complete"
    finally:
        if feeder is not None:
            feeder.close()

    log(f"Stopped: {stop_reason or 'stage complete'}. Steps trained this run: {C.total_step - steps_at_start}")
    if C.total_step == steps_at_start and stop_reason != "stage complete":
        log("!!! NO TRAINING STEPS RAN - see the lines above for the reason.")
    if stop_reason == "time budget" or (stop_reason == "memory" and not IN_SUPERVISOR):
        log(f"NEXT SESSION: attach this notebook's Output as an input dataset and run again with STAGE={stage} "
            f"(it resumes automatically).")
    elif stop_reason == "stage complete":
        log(f"STAGE {stage} COMPLETE. Next: set CFG.STAGE = {min(3, stage + 1)} in a new session with this Output attached.")
    return stop_reason or "stage complete"


## RUN
Trains the selected stage with one process per GPU. Pauses cleanly at `MAX_SESSION_HOURS` and writes `ckpt_latest.pth`.

In [15]:
def _ddp_marker(port, rank):
    return f"/tmp/lego_ddp_ok_{port}_{rank}"


def _init_dist(rank, world, backend, port):
    """Joins the process group. Returns True on success. Any failure (NCCL cannot talk between the GPUs, port problem, ...) -> False."""
    try:
        if backend == "nccl" and CFG.NCCL_P2P_DISABLE:
            os.environ.setdefault("NCCL_P2P_DISABLE", "1")
            os.environ.setdefault("NCCL_IB_DISABLE", "1")
        torch.cuda.set_device(rank)
        kw = dict(backend=backend, init_method=f"tcp://127.0.0.1:{port}", rank=rank, world_size=world,
                  timeout=datetime.timedelta(minutes=CFG.DDP_INIT_TIMEOUT_MIN))
        try:
            if backend == "nccl":
                dist.init_process_group(device_id=torch.device("cuda", rank), **kw)
            else:
                dist.init_process_group(**kw)
        except TypeError:                                   # older torch: no device_id argument
            dist.init_process_group(**kw)
        DIST.on, DIST.backend = True, backend
        probe = torch.ones(1, device=_flag_dev())
        dist.all_reduce(probe)                              # first collective = builds the NCCL communicator
        if int(probe.item()) != world:
            raise RuntimeError(f"all-reduce returned {probe.item()} instead of {world}")
        open(_ddp_marker(port, rank), "w").close()          # tells the supervisor 'the GPUs found each other'
        try:                                                # now allow long gaps (validation / saving on rank 0)
            dist.distributed_c10d._set_pg_timeout(datetime.timedelta(minutes=CFG.DDP_TIMEOUT_MIN))
        except Exception:
            pass
        return True
    except BaseException:
        log(f"DDP start with backend '{backend}' FAILED:\n" + traceback.format_exc()[-1500:], all_ranks=True)
        return False


def _rank_entry(rank, world, backend, attempt, port):
    """One training process (one per GPU). Exit codes: 0 done | 75 restart me (memory) | 76 GPUs could not join | 1 error."""
    global IN_SUPERVISOR
    IN_SUPERVISOR = True
    code = 1
    try:
        sys.stdout = os.fdopen(os.dup(1), "w", buffering=1)      # write to the real stdout (the notebook's pipe)
        sys.stderr = os.fdopen(os.dup(2), "w", buffering=1)
        os.environ["LEGO_RESTART"] = "0" if attempt == 0 else "1"
        DIST.rank, DIST.world, DIST.local, DIST.attempt = rank, world, rank, attempt
        DIST.on, DIST.backend = False, None
        if world > 1 and not _init_dist(rank, world, backend, port):
            code = 76
        else:
            reason = main()
            code = 75 if reason == "memory" else 0
    except BaseException:
        log("!!! RUN FAILED - REASON BELOW !!!\n" + traceback.format_exc(), all_ranks=True)
        code = 1
    finally:
        try:
            sys.stdout.flush(); sys.stderr.flush()
        finally:
            os._exit(code)


def _count_gpus():
    """GPUs visible to the machine. nvidia-smi only: this (parent) process must NEVER touch CUDA before it forks."""
    try:
        r = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, timeout=30)
        return sum(1 for l in r.stdout.splitlines() if l.strip().startswith("GPU"))
    except Exception:
        return 0


def _free_port():
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]


def _wait_all(procs):
    """Waits for all rank processes. If one dies with an error the others are given 20 s to follow, then killed
    (a surviving rank would otherwise sit in an all-reduce for the whole timeout)."""
    while True:
        time.sleep(2)
        codes = [p.exitcode for p in procs]
        if all(c is not None for c in codes):
            return codes
        if any(c not in (None, 0, 75) for c in codes):
            t_end = time.time() + 20
            while time.time() < t_end and any(p.exitcode is None for p in procs):
                time.sleep(1)
            for p in procs:
                if p.exitcode is None:
                    p.terminate()
            for p in procs:
                p.join(10)
                if p.exitcode is None:
                    p.kill()
                    p.join(5)
            return [p.exitcode for p in procs]


def run_supervised():
    """Training runs in forked child processes (one per GPU). Restart on memory request (75); if the GPUs cannot join
    (76) the next backend is tried and finally ONE GPU; any other failure stops with the real traceback."""
    ctx = multiprocessing.get_context("fork")        # nothing has touched CUDA in THIS process, so fork is safe
    n_gpu = min(_count_gpus(), CFG.MAX_GPUS)
    plans = [(n_gpu, b) for b in CFG.DDP_BACKENDS] if (CFG.USE_DDP and n_gpu > 1) else []
    plans.append((1, None))
    plan_i, attempt, restarts, code = 0, 0, 0, 0
    while True:
        world, backend = plans[plan_i]
        log(f"=== launching {world} training process(es)" + (f" [{backend}]" if world > 1 else " [single GPU]")
            + (f" (restart #{restarts})" if restarts else "") + " ===")
        t0 = time.time()
        port = _free_port()
        procs = [ctx.Process(target=_rank_entry, args=(r, world, backend, attempt, port)) for r in range(world)]
        for p in procs:
            p.start()
        codes = _wait_all(procs)
        failed = [c for c in codes if c not in (0, 75)]
        markers = [_ddp_marker(port, r) for r in range(world)]
        ddp_was_up = world == 1 or all(os.path.exists(m) for m in markers)
        for m in markers:
            try:
                os.remove(m)
            except OSError:
                pass
        if (76 in codes or (failed and not ddp_was_up)) and plan_i + 1 < len(plans):
            plan_i += 1
            attempt += 1
            log(f"!!! the GPUs could not start together ({backend}, exit codes {codes}) -> trying "
                f"{plans[plan_i][1] or 'a single GPU'}")
            continue
        if not failed and 75 in codes and hours() < CFG.MAX_SESSION_HOURS - 0.1 and restarts < 60 and time.time() - t0 > 180:
            attempt += 1
            restarts += 1
            log(f"=== training processes restarted (#{restarts}) to release memory - resuming from the last checkpoint ===")
            continue
        code = max([c for c in codes if c] or [0])
        break
    return code


if __name__ == "__main__":
    code = 0
    try:
        if CFG.SUPERVISED:
            code = run_supervised()
        else:
            main()
    except BaseException:
        log("!!! RUN FAILED - REASON BELOW !!!\n" + traceback.format_exc())
        raise
    finally:
        for name in (f"LEGO_s{CFG.STAGE}_best.pth", f"LEGO_s{CFG.STAGE}_latest.pth"):
            src = os.path.join(CFG.WORK_DIR, name)
            if os.path.exists(src):
                try:
                    shutil.copyfile(src, os.path.join(CFG.WORK_DIR, "LEGO.pth"))
                except OSError as e:
                    log(f"could not copy LEGO.pth ({str(e)[:80]})")
                break
        log("Done. Outputs: LEGO.pth, LEGO_latest.pth, ckpt_latest.pth, metrics.csv, samples/")
    if code not in (0,):
        raise RuntimeError(f"training process ended with exit code {code} (75 = memory restart limit reached; "
                           f"76 = GPUs could not start together; 1 = error, see the traceback above)")


[13:52:48] (0.00h) === launching 2 training process(es) [nccl] ===
[13:52:50] (0.00h) === LEGO v5.0 | STAGE 2 | DDP x2 (nccl) | batch 4 per GPU = 8 global | AMP=True ===
[13:52:50] (0.00h) !!! global batch is 8 (not CFG.BATCH_PATCHES=16): one T4 holds at most 4 patches. Steps/epoch and the LR schedule are unchanged, each step sees fewer patches.
[13:53:03] (0.00h) ---------------- PREFLIGHT ----------------
[13:53:03] (0.00h) python 3.13.15 | torch 2.11.0+cu128 | torchvision 0.26.0+cu128 | cv2 4.14.0 | CPUs 4
[13:53:03] (0.00h) GPU 0: Tesla T4 14.6 GB
[13:53:03] (0.00h) GPU 1: Tesla T4 14.6 GB
[13:53:03] (0.00h) RAM (container reader): 1.1/30.0 GB | 29.0 GB free of host
[13:53:03] (0.00h) /kaggle/input folders: ['/kaggle/input/', '/kaggle/input/datasets', '/kaggle/input/datasets/soumikrakshit', '/kaggle/input/datasets/soumikrakshit/div2k-high-resolution-images', '/kaggle/input/datasets/navneetkumargupta', '/kaggle/input/datasets/navneetkumargupta/okokokv4']
[13:53:03] (0.00h) usable im